 ## 0 · Imports & Global Settings

*Loads all external dependencies and defines experiment-wide constants.*

- **System / utility** modules (file I/O, regex, timing, seeding, etc.).
- **Scientific stack**: `numpy`, `pandas`, and `matplotlib`.
- **Forecasting models**
  - Classical ML: `MLPRegressor`, `RandomForestRegressor`, `XGBRegressor`.
  - Time-series: `SARIMAX` (via `statsmodels`) and automated order search with `pmdarima`.
- **Explainability**: local (LIME) and global/local (SHAP) attribution libraries.
- **LLM tooling**: LangChain wrappers for Azure OpenAI, Ollama (Llama-3), and DeepSeek.
- **Misc. utilities**: public-holiday calendar (`holidays`), serialization (`joblib`).

Experiment-level hyper-parameters:

| Symbol | Meaning | Value |
|--------|---------|-------|
| `SEED` | master random seed | `42` |
| `N_LAGS` | number of weekly lag features | `7` |
| `TEST_R` | test-set proportion | `0.30` |
| `RUN_MODE` | quick “test” vs full “prod” pipeline | `"test"` |

The NumPy random seed is initialised (additional libraries set their own seeds later), and all warnings are suppressed to keep notebook output tidy.


In [ ]:
# =========================================================================== #
# 0. Imports & Global Settings                                                #
# =========================================================================== #

# -- System and Utility Modules
import os
import re
import ssl
import time
import math
import json
import shutil
import random
import zipfile
import pprint
import urllib.request
from textwrap import dedent
from datetime import datetime
from pathlib import Path
import warnings
import joblib

# -- Scientific and Data Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# -- Machine Learning
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pmdarima as pm

# -- Explainability Tools
import shap
from lime.lime_tabular import LimeTabularExplainer

# -- NLP and LLM Tools
from langchain.schema import SystemMessage, HumanMessage, AIMessage
from langchain.chat_models import AzureChatOpenAI, ChatOpenAI
from langchain_azure_ai.chat_models import AzureAIChatCompletionsModel

# -- Calendar / Date Utilities
import holidays

# -- OpenAI API
from openai import RateLimitError

# -- Alternate Aliased Imports (if used separately)
import pandas as _pd
import json as _json

# -- Tokens
import tiktoken

# -- Global Settings
SEED    = 42
N_LAGS  = 7
TEST_R  = 0.30
np.random.seed(SEED)
warnings.filterwarnings("ignore")

RUN_MODE = "prod"      # <<<  set to "prod" for full pipeline


## 1 · Data download & weekly feature engineering

1. **Fetch raw data**
   - Downloads the UCI *Household Power Consumption* ZIP (≈ 20 MB) once and extracts `household_power_consumption.txt`.

2. **Parse & clean** (`load_weekly`)
   - Combines `Date` + `Time` into a `DateTime` index.
   - Converts minute-level *kW* readings to *kWh* (`value × 1/60`).
   - Resamples to **weekly totals**.

3. **Calendar features**
   - `weekofyear` (ISO week number).
   - `holiday_week_count`: number of French public holidays inside each week (using the `holidays` package).

4. **Lag features**
   - Adds `lag_1 … lag_7` of the weekly target and drops resulting `NaN` rows.

5. **Train/test split**
   - Chronological split with 70 % train / 30 % test.

6. **Feature matrices**
   - `FEATURES = [lag_1 … lag_7, weekofyear, holiday_week_count]`
   - `EXOG_ONLY_COLS = ["holiday_week_count"]` (used later for SARIMAX).

In [ ]:
# =========================================================================== #
# 1. Data download (≈ 20 MB) & weekly feature engineering                     #
# =========================================================================== #
URL      = ("https://archive.ics.uci.edu/ml/machine-learning-databases/00235/"
            "household_power_consumption.zip")
ZIP_F    = Path("household_power_consumption.zip")
TXT_F    = Path("household_power_consumption.txt")

if not TXT_F.exists():
    if not ZIP_F.exists():                  # download once
        print("Downloading dataset …")
        with urllib.request.urlopen(URL, context=ssl._create_unverified_context()) as r,\
             open(ZIP_F, "wb") as f:
            shutil.copyfileobj(r, f)
    with zipfile.ZipFile(ZIP_F) as zf:      # unzip
        zf.extractall(".")

def load_weekly(file_path: str, country="FR") -> pd.DataFrame:
    df = pd.read_csv(
        file_path, sep=";", dayfirst=True, na_values="?", low_memory=False
    )
    df["DateTime"] = pd.to_datetime(df["Date"] + " " + df["Time"], dayfirst=True)
    df.set_index("DateTime", inplace=True)

    # kW-min  →  kWh per minute
    df["Global_active_power"] = (
        df["Global_active_power"].astype(float).interpolate() * (1 / 60)
    )

    # ------------------------------------------------------------------ #
    # WEEKLY AGGREGATION (anchor = **Monday**, matches ISO week numbers)
    # ------------------------------------------------------------------ #
    w = (
        df["Global_active_power"]
        .resample("W-MON")          # <-- change from "W" to "W-MON"
        .sum()
        .to_frame("True_Value")
        .reset_index()
    )

    # ISO-8601 week index (1-52, occasionally 53)
    w["weekofyear"] = w["DateTime"].dt.isocalendar().week.astype(int)

    # Holiday count inside each ISO week
    fr_hols = holidays.CountryHoliday(country)

    def hol_cnt(week_end):
        wk_start = week_end - pd.Timedelta(days=6)
        return sum(
            (wk_start + pd.Timedelta(days=i)).date() in fr_hols for i in range(7)
        )

    w["holiday_week_count"] = w["DateTime"].apply(hol_cnt)
    return w


weekly = load_weekly(TXT_F)

# lag features
for i in range(1, N_LAGS+1):
    weekly[f"lag_{i}"] = weekly["True_Value"].shift(i)
weekly = weekly.dropna().reset_index(drop=True)

cut      = int(len(weekly)*(1-TEST_R))
train, test = weekly.iloc[:cut], weekly.iloc[cut:]

FEATURES       = [f"lag_{i}" for i in range(1, N_LAGS+1)] + ["weekofyear","holiday_week_count"]
EXOG_ONLY_COLS = ["holiday_week_count"]

X_train, y_train = train[FEATURES], train["True_Value"]
X_test,  y_test  = test[FEATURES],  test["True_Value"]

In [ ]:
X_train

In [ ]:
y_train

In [ ]:
X_test

In [ ]:
y_test

## 2 · XGBoost hyperparameter tuning

We perform a **random search** (200 trials) over XGBoost hyperparameters, evaluated via **walk-forward cross-validation**:

1. **Hyperparameter ranges**
   - `n_estimators`: [1 000, 5 000]
   - `learning_rate`: [0.002, 0.02]
   - `max_depth`: [4, 11]
   - `subsample`, `colsample_bytree`: [0.7, 1.0]
   - `min_child_weight`: [1, 6]
   - `gamma`: [0.0, 0.5]
   - `reg_lambda`: [0.1, 2.0]

2. **Sampling function** (`_sample_params`) draws one random configuration per trial.
   *Note*: For full reproducibility, we seed Python’s `random` module as well.

3. **Walk-forward CV**
   - `TimeSeriesSplit(n_splits=4, test_size=⌈0.15·|X_train|⌉)`
   - Each fold trains on past data and validates on the subsequent slice.

4. **Model fitting**
   - `XGBRegressor(..., objective="reg:squarederror", tree_method="hist")`
   - Early stopping after 200 rounds on validation MAE.

5. **Selection criterion**
   - Compute the mean validation MAE across folds.
   - Retain the parameter set with the lowest CV‐MAE in `XGB_BEST_PARAMS`.

> **Compute time:** ≈ 20 minutes on a modest CPU (weekly data, ~100 training points).


In [ ]:
print("\n🛠  XGB tuning: 200 random draws × walk-forward CV (this may take ~20 min)")

# --------------------------------- hyper-param ranges -------------------
_param_space = {
    "n_estimators"     : (1000, 5000),
    "learning_rate"    : (0.002, 0.02),
    "max_depth"        : (4, 11),
    "subsample"        : (0.7, 1.0),
    "colsample_bytree" : (0.7, 1.0),
    "min_child_weight" : (1, 6),
    "gamma"            : (0.0, 0.5),
    "reg_lambda"       : (0.1, 2.0),
}

def _sample_params():
    return {
        "n_estimators"     : random.randint(*_param_space["n_estimators"]),
        "learning_rate"    : random.uniform(*_param_space["learning_rate"]),
        "max_depth"        : random.randint(*_param_space["max_depth"]),
        "subsample"        : random.uniform(*_param_space["subsample"]),
        "colsample_bytree" : random.uniform(*_param_space["colsample_bytree"]),
        "min_child_weight" : random.randint(*_param_space["min_child_weight"]),
        "gamma"            : random.uniform(*_param_space["gamma"]),
        "reg_lambda"       : random.uniform(*_param_space["reg_lambda"]),
        # fixed knobs
        "objective"        : "reg:squarederror",
        "random_state"     : SEED,
        "n_jobs"           : -1,
        "tree_method"      : "hist",
        "verbosity"        : 0,
    }

tscv       = TimeSeriesSplit(n_splits=4, test_size=math.ceil(len(X_train)*0.15))
N_ITER     = 200
best_mae   = np.inf
best_param = None
t0         = time.time()

for i in range(1, N_ITER+1):
    p   = _sample_params()
    mae = []

    for tr_idx, val_idx in tscv.split(X_train):
        X_tr, X_val = X_train.iloc[tr_idx][FEATURES], X_train.iloc[val_idx][FEATURES]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        m = XGBRegressor(**p)
        m.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="mae",
            early_stopping_rounds=200,
            verbose=False,
        )
        mae.append(m.evals_result()["validation_0"]["mae"][-1])

    cv_mae = np.mean(mae)
    if cv_mae < best_mae:
        best_mae, best_param = cv_mae, p

    if i % 10 == 0 or i == 1:
        print(f"  {i:3d}/{N_ITER}  cv-MAE {cv_mae:5.2f}   best {best_mae:5.2f}")

# --------------------------- save result for later -----------------------
XGB_BEST_PARAMS = best_param          # ← global variable you’ll reuse
print("\n🏁  Tuning complete.  Best cv-MAE = {:.2f}".format(best_mae))
print("    XGB_BEST_PARAMS:")
pprint.pp(XGB_BEST_PARAMS)

## 3 · Automated SARIMAX order selection

We use **pmdarima.auto_arima** to identify optimal SARIMAX orders:

1. **Model specification**
   - Endogenous: weekly global active power (`y_train`).
   - Exogenous regressor: `holiday_week_count`.
   - Seasonal component: `m=52` (annual cycle).

2. **Search configuration**
   - Non-seasonal AR/MA orders (`p`, `q`): 0–3.
   - Seasonal AR/MA orders (`P`, `Q`): 0–2.
   - Seasonal differencing forced (`D=1`), non-seasonal differencing (`d`) determined automatically.
   - Transformation: allow optional log-transform of data.
   - Information criterion: **AICc**, using a stepwise heuristic (`stepwise=True`).
   - Seasonality test: **Canova–Hansen**.
   - Parallelized (`n_jobs=-1`), warnings suppressed.

3. **Results**
   - The best model orders `(p,d,q)` and `(P,D,Q,52)` are stored in `SARIMAX_KWARGS` for subsequent fitting.



In [ ]:
print("🔍  Running auto_arima order search …")

auto_model = pm.auto_arima(
    y_train,                    # target
    exogenous=X_train[EXOG_ONLY_COLS],  # holiday dummy
    seasonal=True, m=52,        # yearly seasonality on weekly data
    start_p=0, max_p=3,         # AR terms
    start_q=0, max_q=3,         # MA terms
    start_P=0, max_P=2,         # seasonal AR
    start_Q=0, max_Q=2,         # seasonal MA
    d=None, D=1,                # let auto_arima decide non-seasonal diff; force seasonal diff=1
    seasonal_test="ch",         # Canova–Hansen → robust seasonality test
    stepwise=True,              # fast heuristic search
    n_jobs=-1,
    information_criterion="aicc",
    transform="log",            # allow log-transform if it improves fit
    suppress_warnings=True,
    error_action="ignore",
    trace=True,                 # print tried combos
)

SARIMAX_ORDER, SARIMAX_SORDER = auto_model.order, auto_model.seasonal_order
print(f"✅  Selected order={SARIMAX_ORDER}, seasonal_order={SARIMAX_SORDER}")

# Store for later use in the model-zoo block
SARIMAX_KWARGS = dict(
    order=SARIMAX_ORDER,
    seasonal_order=SARIMAX_SORDER,
    enforce_stationarity=False,
    enforce_invertibility=False,
    trend='c'
)

In [ ]:
SARIMAX_ORDER   = (2, 0, 1)
SARIMAX_SORDER  = (1, 1, 0, 52)
print(f"Using fixed order={SARIMAX_ORDER}, seasonal_order={SARIMAX_SORDER}")

SARIMAX_KWARGS = dict(
    order=SARIMAX_ORDER,
    seasonal_order=SARIMAX_SORDER,
    enforce_stationarity=False,
    enforce_invertibility=False,
    trend='c'
)

## 4 · Model training, rolling forecast, and evaluation

### 4.1 · Machine-learning models
- **Models**
  - **MLP**: single hidden layer (8 units), tanh activation, SGD solver, 30 iterations.
  - **Random Forest**: 10 trees, depth=1 (stumps), min_samples_leaf=25, max_features=8 %.
  - **XGBoost**: best hyperparameters from the previous random search.
- **Procedure**
  1. Fit each model on the full training set (`X_train`, `y_train`).
  2. Predict on the test set (`X_test`).
  3. Compute **MAE**, **RMSE**, and **R²**.

### 4.2 · SARIMAX rolling one-step forecasts

To match the ML models’ one-step-ahead paradigm, we use a **stateful** forecast loop:

1. **Initial fit** on `y_train` and `holiday_week_count` with orders = `(p,d,q)` and `(P,D,Q,52)`.
2. **For each test week**:
   - Forecast **one** week ahead (`get_forecast(steps=1, exog=…)`).
   - **Append the actual observed** test value back into the model state (`append(..., refit=False)`), so the next forecast always uses the true last value.
3. Compute MAE, RMSE, and R² over these rolling one-step predictions.

This approach ensures that all four models—MLP, RandomForest, XGBoost, and SARIMAX—make strictly one-step ahead forecasts using the same informational inputs (true past lags and holiday counts), enabling fair comparison.


### 4.3 · Results
- **Metrics table**: test-set performance for all four models (sorted by MAE).
- **Visualization**: overlay of true vs. predicted weekly power consumption across models.


In [ ]:
# ======================================================================= #
# Train models (MLP, RF, XGB) + rolling-update SARIMAX                #
#       -- identical FEATURES for every ML model --                      #
# ======================================================================= #

# ── deliberately different capacities, SAME FEATURES ──────────────────
# ── machine-learning models: bad → mid → good ────────────────────────────
model_zoo = {
    # ─── BAD ───  very weak MLP ________________________________________
    "MLP": make_pipeline(
        StandardScaler(),
        MLPRegressor(
            hidden_layer_sizes=(8,),
            activation="tanh",
            solver="sgd",
            learning_rate_init=0.30,
            momentum=0.0,
            max_iter=30,
            n_iter_no_change=2,
            alpha=0.0,
            random_state=SEED,
        ),
    ),

    # ─── MID ───  Random Forest tuned for ≈ 0.30 R² _____________________
    "RandomForest": RandomForestRegressor(
        n_estimators     = 10,    # still tiny
        max_depth        = 1,     # stump-level depth
        min_samples_leaf = 25,
        max_features     = 0.08,  # 8 % of predictors per split
        bootstrap        = True,
        random_state     = SEED,
    ),

    # ─── GOOD ───  your best XGB params (≈ 0.62 R²) _____________________
    "XGB": XGBRegressor(
        n_estimators     = 1190,
        learning_rate    = 0.00497305889913237,
        max_depth        = 5,
        subsample        = 0.8047436120859401,
        colsample_bytree = 0.7253704053768965,
        min_child_weight = 6,
        gamma            = 0.24843151643153033,
        reg_lambda       = 0.5295892281900806,
        objective        = "reg:squarederror",
        random_state     = 42,
        n_jobs           = -1,
        tree_method      = "hist",
        verbosity        = 0,
    ),
}

preds, metrics = {}, {}

# ── fit + predict on identical feature set ────────────────────────────
for name, mdl in model_zoo.items():
    mdl.fit(X_train[FEATURES], y_train)
    p = mdl.predict(X_test[FEATURES])
    preds[name] = p
    metrics[name] = {
        "MAE" : mean_absolute_error(y_test, p),
        "RMSE": np.sqrt(mean_squared_error(y_test, p)),   # manual √MSE
        "R2"  : r2_score(y_test, p),
    }

# ── classical SARIMAX with rolling 1-step-ahead updates ───────────────
sarimax = SARIMAX(
    endog=y_train,
    exog=train[EXOG_ONLY_COLS],
    **SARIMAX_KWARGS,               # use the selected (p,d,q) & (P,D,Q,52)
).fit(disp=False)

y_test_ext      = y_test.copy()
exog_test_ext   = test[EXOG_ONLY_COLS].copy()
start_idx       = len(y_train)
y_test_ext.index    = np.arange(start_idx, start_idx + len(y_test))
exog_test_ext.index = y_test_ext.index

stateful_res = sarimax
sarimax_one_step = []
for idx in y_test_ext.index:
    exog_row = exog_test_ext.loc[[idx]]
    pred_val = stateful_res.get_forecast(steps=1, exog=exog_row)\
                           .predicted_mean.iloc[0]
    sarimax_one_step.append(pred_val)
    stateful_res = stateful_res.append(
        endog=y_test_ext.loc[[idx]],
        exog=exog_row,
        refit=False,
    )

sarimax_one_step = np.array(sarimax_one_step)
preds["SARIMAX"] = sarimax_one_step
metrics["SARIMAX"] = {
    "MAE" : mean_absolute_error(y_test, sarimax_one_step),
    "RMSE": np.sqrt(mean_squared_error(y_test, sarimax_one_step)),
    "R2"  : r2_score(y_test, sarimax_one_step),
}

# ── tidy metrics table ────────────────────────────────────────────────
metrics_df = (
    pd.DataFrame(metrics)
    .T.round(2)
    .sort_values("MAE")
    .rename_axis("Model")
)
print("\nTest-set performance:")
display(metrics_df)

# ── plot true series vs all forecasts ──────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))      # keep handle
ax.plot(test["DateTime"], y_test, label="True", linewidth=2, c="k")
for name, p in preds.items():
    ax.plot(test["DateTime"], p, "--", label=name)

ax.set(title="Weekly household power – true vs model predictions",
       xlabel="Week ending", ylabel="kWh")
ax.legend()
fig.tight_layout()
plt.show()


## 5 · Persisting results and artifacts

All key outputs are saved under `outputs_final/`:

1. **Performance metrics**
   - `metrics_summary.csv`: table of MAE, RMSE, R² for each model.

2. **Model predictions**
   - `{ModelName}_predictions.csv` for MLP, RandomForest, XGB, SARIMAX, each containing:
     `DateTime`, `True_Value`, and `Prediction`.

3. **Test features & targets**
   - `test_features_targets.csv`: the `FEATURES` columns plus `True_Value` on the test split, for any later XAI or analysis.

4. **Model artefacts**
   - Serialized models via `joblib` / `pickle`:
     - `MLP_model.joblib`
     - `RandomForest_model.joblib`
     - `XGB_model.joblib`
     - `SARIMAX_model.pkl`

5. **Visualization**
   - `forecast_comparison.pdf`: hi-res PDF of true vs. predicted weekly series for all models.

6. **SARIMAX internals**
   - `sarimax_coefficients.csv`: estimated model coefficients.
   - `sarimax_summary.txt`: full printed summary of the fitted SARIMAX model.

> **Optional**: Uncomment the JSON dump for `XGB_BEST_PARAMS` to persist the exact hyperparameters used.


In [ ]:
# %%  ────────────────────────────────────────────────────────────────────
# Save all required artifacts (metrics, preds, models, figure as PDF)
# ───────────────────────────────────────────────────────────────────────

OUT_DIR = Path("outputs_final")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1) metrics table
metrics_df.to_csv(OUT_DIR / "metrics_summary.csv", index=True)

# 2) per-model prediction CSVs
for model_name, arr in preds.items():
    pd.DataFrame({
        "DateTime"  : test["DateTime"].values,
        "True_Value": y_test.values,
        "Prediction": arr,
    }).to_csv(OUT_DIR / f"{model_name}_predictions.csv", index=False)

# 3) test features (+ target) for later XAI work
test.to_csv(OUT_DIR / "test_features_targets.csv", index=False)

# 4) best XGB hyper-parameters
# with open(OUT_DIR / "xgb_best_params.json", "w") as f:
#     json.dump(XGB_BEST_PARAMS, f, indent=2)

# 5) persist trained models
joblib.dump(model_zoo["MLP"],          OUT_DIR / "MLP_model.joblib")
joblib.dump(model_zoo["RandomForest"], OUT_DIR / "RandomForest_model.joblib")
joblib.dump(model_zoo["XGB"],          OUT_DIR / "XGB_model.joblib")
joblib.dump(sarimax,                   OUT_DIR / "SARIMAX_model.pkl")

# 6) save the forecast figure as a hi-res PDF
fig_pdf = OUT_DIR / "forecast_comparison.pdf"
fig.savefig(fig_pdf, dpi=300, bbox_inches="tight")
print(f"📈  Figure saved → {fig_pdf}")

# (optional) also keep a PNG for quick preview
# fig_png = OUT_DIR / "forecast_comparison.png"
# fig.savefig(fig_png, dpi=300, bbox_inches="tight")

# 7) SARIMAX coefficients & summary (CSV + txt only, no PDF)
sarimax.params.to_frame(name="Coefficient").round(4)\
      .to_csv(OUT_DIR / "sarimax_coefficients.csv")
with open(OUT_DIR / "sarimax_summary.txt", "w") as f:
    f.write(str(sarimax.summary()))

print(f"✅  Everything saved to {OUT_DIR.resolve()}")


## 6 · Selecting representative test weeks for narrative examples

To build a concise “storyline” of model behavior, we choose exactly three test weeks—one each from early, middle, and late periods—where the forecasting errors strictly rank:

XGB error < SARIMAX error < RandomForest error < MLP error


1. **Error matrix**
   - Compute absolute errors `|prediction – true|` for each model at every test index.

2. **Ranking filter**
   - Identify all indices where sorting these errors yields `["XGB","SARIMAX","RandomForest","MLP"]`.

3. **Tercile selection**
   - Split the test set (N weeks) into three equal-length terciles:
     - Early: indices `[0, N/3)`
     - Middle: `[N/3, 2N/3)`
     - Late: `[2N/3, N)`
   - In each tercile, pick the **earliest** qualifying index.
   - If any tercile lacks a qualifying index, fill via the next unused qualifying index.

4. **Build summary table**
   - Create `story_points_df` with 12 rows (3 weeks × 4 models), containing:
     - `RowPos`, `DateTime`, `Model`, `True_Value`, `Prediction`, `Pct_Error`.

5. **Dictionary lookup**
   - Assemble `STORY_POINTS[model]` for quick retrieval of the three example records per model.

The resulting weeks serve as consistent, interpretable examples when generating model explanations in the paper.


In [ ]:
# %%  ────────────────────────────────────────────────────────────────────
# Pick 3 rows (early / mid / late) where abs-error ranking is
#   XGB  <  SARIMAX  <  RandomForest  <  MLP
# They form a tidy storyline for the paper.
# ───────────────────────────────────────────────────────────────────────
order = ["XGB", "SARIMAX", "RandomForest", "MLP"]

# 1) Compute absolute errors for every row & model
err_mat = {
    mdl: np.abs(preds[mdl] - y_test.values)
    for mdl in order
}
err_df = pd.DataFrame(err_mat)            # rows = test positions

# 2) helper: does this row follow the desired ranking?
def is_rank_match(row):
    return list(row.sort_values().index) == order

qualifying_idx = [i for i, row in err_df.iterrows() if is_rank_match(row)]

if len(qualifying_idx) < 3:
    raise ValueError("Couldn’t find 3 distinct rows with the desired ranking. "
                     "Loosen the criterion or inspect err_df.")

n_test = len(err_df)
third  = n_test // 3

# 3) choose earliest qualifying row in each tercile
row_sel = []
for start, end in [(0, third), (third, 2*third), (2*third, n_test)]:
    rows_in_band = [i for i in qualifying_idx if start <= i < end]
    if rows_in_band:
        row_sel.append(rows_in_band[0])         # first match in that band

# fallback: if a tercile had none, just take next unused qualifying row
unused = [i for i in qualifying_idx if i not in row_sel]
while len(row_sel) < 3 and unused:
    row_sel.append(unused.pop(0))

# 4) build the dataframe
story_records = []
for pos in row_sel:
    base = test.iloc[pos]
    for mdl in order:
        pred_val = preds[mdl][pos]
        true_val = y_test.iloc[pos]
        pct_err  = (pred_val - true_val) / true_val * 100
        story_records.append({
            "RowPos"    : pos,
            "DateTime"  : base["DateTime"],
            "Model"     : mdl,
            "True_Value": true_val,
            "Prediction": pred_val,
            "Pct_Error" : round(pct_err, 2),
        })

story_points_df = (pd.DataFrame(story_records)
                   .sort_values(["RowPos", "Model"])
                   .reset_index(drop=True))

# dict version for quick look-ups later
STORY_POINTS = {
    mdl: story_points_df[story_points_df["Model"] == mdl].to_dict(orient="records")
    for mdl in order
}

print("\nStoryline points (same dates, error order XGB < … < MLP):")
display(story_points_df)


## 7 · Computing SHAP explanations for ML models

We generate **Shapley‐value attributions** for each machine‐learning model to quantify feature contributions:

1. **Initialization**
   - Call `shap.initjs()` to enable potential JS visualizations.

2. **Explainer selection**
   - **MLP**: `KernelExplainer`
     - Background sample: 100 randomly‐selected training rows (`shap.sample`, `random_state=SEED`).
     - Prediction wrapper: model’s `.predict` on DataFrame inputs.
   - **RandomForest** & **XGBoost**: `TreeExplainer` (exact, fast for tree models).

3. **SHAP value extraction**
   - `values`: array of shape `(n_test × n_features)`.
   - `expected_value`: model’s base prediction.

4. **Storage**
   - `SHAP_INFO[model] = {explainer, values, base_value}` for downstream prompt generation and analysis.

5. **Convenience DataFrame**
   - `shap_values_df`: concatenated SHAP values with MultiIndex `(Model, TestRow)` for quick inspection.

> **Optional**: Save `shap_values_df.to_csv(...)` here to avoid re-running this expensive computation later.


In [ ]:
# %%  ────────────────────────────────────────────────────────────────────
# Compute SHAP values for each ML model (skip SARIMAX)                   #
# ───────────────────────────────────────────────────────────────────────
shap.initjs()

SHAP_INFO = {}

for mdl_name, mdl in model_zoo.items():
    if mdl_name == "SARIMAX":
        continue  # classical model → no SHAP

    print(f"⚙️  Computing SHAP for {mdl_name} …")

    if mdl_name == "MLP":
        # KernelExplainer over a background sample
        background = shap.sample(X_train[FEATURES], 100, random_state=SEED)
        predict_fn = lambda arr: mdl.predict(pd.DataFrame(arr, columns=FEATURES))
        explainer  = shap.KernelExplainer(predict_fn, background.values)
        shap_vals  = explainer.shap_values(X_test[FEATURES].values)
        shap_vals  = shap_vals[0] if isinstance(shap_vals, list) else shap_vals
        base_val   = explainer.expected_value
        if isinstance(base_val, (list, np.ndarray)):
            base_val = base_val[0]
    else:
        # TreeExplainer for RF & XGB
        explainer = shap.TreeExplainer(mdl)
        shap_vals = explainer.shap_values(X_test[FEATURES])
        shap_vals = shap_vals[0] if isinstance(shap_vals, list) else shap_vals
        base_val  = explainer.expected_value
        if isinstance(base_val, (list, np.ndarray)):
            base_val = base_val[0]

    SHAP_INFO[mdl_name] = {
        "explainer"  : explainer,
        "values"     : shap_vals,    # numpy array shape (n_test, n_features)
        "base_value" : base_val,
    }

print("\n✅  SHAP values stored in SHAP_INFO")

# (Optional) convenience dataframe for quick inspection
shap_values_df = (
    pd.concat({
        mdl: pd.DataFrame(vals["values"], columns=FEATURES)
        for mdl, vals in SHAP_INFO.items()
    }, names=["Model"])
)

In [ ]:
# Example access:
SHAP_INFO["XGB"]["values"][0]          # SHAP vector for first test row


In [ ]:
SHAP_INFO["XGB"]["explainer"]

In [ ]:
SHAP_INFO["XGB"]["base_value"]         # expected value

In [ ]:
shap_values_df                         # dataframe view (can be huge)

## 8 · Computing LIME explanations for ML models

We generate **LIME** local explanations for each ML model on every test instance:

1. **Hyper-parameters**
   - `kernel_width = 3` (neighbourhood kernel).
   - `num_samples = 8 000` (synthetic perturbations).

2. **Feature scaling (for the surrogate only)**
   - **Tree models** (RF, XGB): one common `StandardScaler` fit on `X_train`.
   - **MLP**: reuse its internal `StandardScaler` to match the pipeline.
   - *This scaling ensures the surrogate linear model sees features on comparable scales; it does **not** alter the original tree or neural-network predictions, which remain in kWh.*

3. **Explainer factory** (`make_explainer(model_name)`) returns:
   - A `LimeTabularExplainer` trained on the **scaled** training matrix.
   - `to_model` / `to_raw` transforms to convert between raw and scaled spaces for consistent predictions.

4. **Local explanation loop**
   For each model and each test row:
   - Scale the raw features, then call `explainer.explain_instance(...)`.
   - Extract per-feature contributions (Δ kWh) and an `intercept`.
   - Store the contributions in a dense record keyed by `(Model, Row)`.

5. **Data assembly**
   - Combine records into `lime_full_df` (index: `Model, Row`; columns: `lag_1…lag_7`, `weekofyear`, `holiday_week_count`, `intercept`).
   - Save to `LIME_full_contributions_tight.csv` for downstream prompts and analysis.

> **Clarification:** Although the surrogate is trained on z-score–transformed features, the reported contributions from `exp.as_list()` are in the original target units (kWh), since LIME’s linear surrogate is fitted to match the model’s raw predictions.


In [ ]:
# %% ────────────────────────────────────────────────────────────────────
# LIME contributions with proper scaling  →  lime_full_df
# ───────────────────────────────────────────────────────────────────────

OUT_DIR = Path("outputs_final")
OUT_DIR.mkdir(exist_ok=True)

# -------------------------------------------------------------------
# ✨ LIME hyper-parameters you can tune quickly
# -------------------------------------------------------------------
LIME_KERNEL  = 3      # 4–6 worked well in tests
LIME_SAMPLES = 8000   # ↑ synthetic neighbourhood size

# -------------------------------------------------------------------
# 1.  Common z-scaler for *all* tree models (already created earlier)
# -------------------------------------------------------------------
scale_for_lime = StandardScaler().fit(X_train[FEATURES])

# -------------------------------------------------------------------
# 2.  Re-usable explainer factory
# -------------------------------------------------------------------
def make_explainer(model_name, kernel=LIME_KERNEL):
    """
    Returns:
        explainer, to_model_space(), to_raw_space()
    so you can keep the rest of your code unchanged.
    """
    if model_name == "MLP":
        # use the SAME scaler that lives inside the pipeline
        scaler_in_pipe = model_zoo["MLP"].named_steps["standardscaler"]
        train_mat  = scaler_in_pipe.transform(X_train[FEATURES])
        to_model   = scaler_in_pipe.transform
        to_raw     = scaler_in_pipe.inverse_transform
    else:  # RandomForest, XGB
        train_mat  = scale_for_lime.transform(X_train[FEATURES])
        to_model   = scale_for_lime.transform
        to_raw     = scale_for_lime.inverse_transform

    explainer = LimeTabularExplainer(
        training_data         = train_mat,
        feature_names         = FEATURES,
        mode                  = "regression",
        discretize_continuous = False,
        sample_around_instance= True,
        kernel_width          = kernel,
        random_state          = SEED,
    )
    return explainer, to_model, to_raw

# ---------------------------------------------------------------------
# build dense contribution matrix (incl. intercept) for every model/row
# ---------------------------------------------------------------------
records = []
for mdl in ["MLP", "RandomForest", "XGB"]:
    explainer, to_model, to_raw = make_explainer(mdl)

    for r in range(len(X_test)):
        raw_row   = X_test.iloc[r][FEATURES].values.reshape(1, -1)
        row_model = to_model(raw_row)[0]

        # ---------- prediction wrapper (already OK) ----------
        predict_fn = lambda arr, m=mdl, rev=to_raw: model_zoo[m].predict(
            pd.DataFrame(rev(arr), columns=FEATURES)
        )

        # --------- call LIME once, re-use row_model ----------
        exp = explainer.explain_instance(
            data_row     = row_model,          # ✅ use the variable you just built
            predict_fn   = predict_fn,         # ✅ avoid redefining the lambda
            num_features = len(FEATURES),
            num_samples  = LIME_SAMPLES,
        )

        dense = {f: 0.0 for f in FEATURES}
        dense.update(dict(exp.as_list()))      # fills non-zero weights
        dense["intercept"] = exp.intercept[0]
        dense["Model"]     = mdl
        dense["Row"]       = X_test.index[r]
        records.append(dense)

lime_full_df = (
    pd.DataFrame(records)
      .set_index(["Model", "Row"])
      .astype(float)
      .sort_index()
)

# optional: save to disk
lime_full_df.to_csv(OUT_DIR / "LIME_full_contributions_tight.csv")
print("💾  Saved  →", OUT_DIR / "LIME_full_contributions_tight.csv")
display(lime_full_df.head())

## 9 · Sanity check of LIME surrogate fidelity on example weeks

To ensure the LIME surrogate accurately approximates each model locally, we:

1. **Identify storyline rows**
   - Use `row_sel` (or `story_points_df`) to get the three selected test‐week indices.

2. **Define fidelity helper**
   - `local_gap_and_r2(row_idx, model_name, explainer, to_model, to_raw)`:
     - Transforms raw features → scaled space.
     - Calls `explain_instance()` to fit LIME’s local linear surrogate.
     - Retrieves:
       - **Local prediction** (`exp.local_pred[0]`).
       - **Local R²** fidelity score (`exp.score`).
     - Returns:
       - Δ = (true model prediction − surrogate prediction) in kWh.
       - R² indicating surrogate fit quality.

3. **Evaluate each model & row**
   - For **MLP**, **RandomForest**, **XGB**:
     1. Re-create explainer with `make_explainer(model_name)`.
     2. For each `pos` in `storyline_rows`, compute `(Δ, R²)`.
     3. Print results.


In [ ]:
# %% ────────────────────────────────────────────────────────────────────
# Sanity-check on the storyline rows picked above (row_sel / story_points_df)
# ───────────────────────────────────────────────────────────────────────
# 1) collect the storyline row positions (indices into X_test)
try:
    storyline_rows = sorted(set(row_sel))
except NameError:
    storyline_rows = sorted(story_points_df["RowPos"].unique())

print("📌  Storyline RowPos:", storyline_rows, "\n")

# 2) helper from earlier – make_explainer(model_name) must exist
def local_gap_and_r2(row_idx, model_name, explainer, to_model, to_raw):
    raw_row   = X_test.iloc[row_idx][FEATURES].values.reshape(1, -1)
    row_model = to_model(raw_row)[0]

    # ⚠️  use *model_name*, not undefined ‘mdl’
    pred_fn   = lambda arr: model_zoo[model_name].predict(
        pd.DataFrame(to_raw(arr), columns=FEATURES)
    )

    exp = explainer.explain_instance(
        data_row     = row_model,
        predict_fn   = pred_fn,
        num_features = len(FEATURES),
        num_samples  = LIME_SAMPLES,
    )

    model_pred = model_zoo[model_name].predict(X_test.iloc[[row_idx]])[0]
    return model_pred - exp.local_pred[0], exp.score

# 3) run check for each model and storyline row
print("🧪  LIME vs. model on storyline points\n" + "-"*70)
for mdl in ["MLP", "RandomForest", "XGB"]:
    explainer, fwd, rev = make_explainer(mdl)
    for pos in storyline_rows:
        gap, r2 = local_gap_and_r2(pos, mdl, explainer, fwd, rev)
        print(f"{mdl:<13} RowPos {pos:>2}  Δ={gap:+7.2f} kWh   R²={r2:5.3f}")
print("-"*70)
print("Aim for |Δ| ≲ 5 kWh and R² close to 1 for good local fidelity.")


## 10 · Shared helper functions for prompt construction

Before generating LLM prompts for each model and test instance, we define reusable utility functions:

1. **Block formatting**
   - `_block(lines)`: join lines with newline separators.
   - `_explain_block(xai_dict, title, base)`:
     - If no XAI data, returns `""`.
     - Otherwise, sorts features by magnitude, formats each as `name: ±value`, prepends `"{TITLE} values:"`, appends the base prediction.
     - For LIME, adds a note clarifying that weights are “βᵢ × xᵢ” (kWh contributions).

2. **Prompt builder** (`make_prompt`)
   Concatenates text blocks for:
   - **Problem description**: domain, dataset summary.
   - **Model performance**: MAE, RMSE, R².
   - **Prediction**: point forecast.
   - **Instance context**: raw feature values.
   - **XAI explanation**: SHAP or LIME contributions, if provided.

3. **XAI extractors**
   - `_shap_for(model, idx)`: returns the SHAP value dict and base value for test-row `idx`.
   - `_lime_for(model, pos)`: returns the LIME weight dict and intercept for position `pos` (maps position → actual DataFrame index).

4. **Prompt storage**
   - `HUMAN_PROMPTS_ALL = {}` will collect all prompts keyed by `(row_pos, model_name, xai_kind)`.

These helpers ensure consistent, readable prompts for feeding into various LLM strategies.


In [ ]:
 # %%%%%%%%%%%%%%%%%%%% BUILDING HUMAN MESSAGES %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# %% ────────────────────────────────────────────────────────────────────
# Shared helpers  (run once)
# ──────────────────────────────────────────────────────────────────────

def _block(lines):
    return "\n".join(lines) + "\n"

def _explain_block(xai_dict, title, base):
    """
    Build the SHAP / LIME section.
    For LIME we append a clarifying note.
    """
    if not xai_dict:                      # “none”
        return ""

    ordered = sorted(xai_dict.items(), key=lambda kv: abs(kv[1]), reverse=True)
    lines   = [f"{k}: {v:+.3f}" for k, v in ordered]
    block   = [f"{title} values:"] + lines + \
              [f"The expected/base value for {title}: {base:.3f}"]

    if title == "LIME":
        block.append(
            "(Each LIME value is already βᵢ × xᵢ: the additive kWh contribution "
            "of that feature for this instance.)"
        )
    return "\n".join(block) + "\n"

def make_prompt(row_idx, model, perf, features, pred,
                xai_kind, xai_vals, base_val):
    header = _block([
        "The following is about time series data with a single-step ahead prediction, "
        "where the model predicts the next value in the time series based on previous observations.",
        "Data Domain: Energy Consumption",
        "Dataset Description:",
        "- The dataset contains 2,075,259 measurements from a house in Sceaux, France (near Paris), Dec 2006–Nov 2010.",
        "- Recorded at one-minute resolution, then resampled weekly.",
        "- Lag_1..Lag_7, ISO week number and number of public holidays per week were added as features.",
        "- Target: weekly global active power (kWh).",
        ""
    ])

    perf_block = _block([
        f"Model Used: {model}",
        "Model Performance:",
        f"  - MAE: {perf['MAE']:.3f}",
        f"  - RMSE: {perf['RMSE']:.3f}",
        f"  - R²: {perf['R2']:.3f}",
        ""
    ])

    pred_block = _block([f"Prediction: {pred:.2f}", ""])

    feat_lines = [f"{k}: {v}" for k, v in features.items()]
    feat_block = _block(["Instance Features or Context:"] + feat_lines + [""])

    xai_block  = _explain_block(xai_vals, xai_kind.upper(), base_val)

    return dedent(header + perf_block + pred_block + feat_block + xai_block).strip()

# helper accessors  ----------------------------------------------------
def _shap_for(model, idx):
    arr  = SHAP_INFO[model]["values"][idx]
    return dict(zip(FEATURES, arr)), float(SHAP_INFO[model]["base_value"])

def _lime_for(model, pos):
    """
    pos … integer *position* inside X_test  (0-based, what row_sel holds)
    returns dict of LIME weights  +  intercept (base value)
    """
    row_label = X_test.index[pos]          # convert position → label
    row       = lime_full_df.loc[(model, row_label)]
    base      = float(row["intercept"])
    return row.drop("intercept").to_dict(), base

# master container
HUMAN_PROMPTS_ALL = {}


## 11 · Generating text prompts for each model & XAI mode

For each of the three selected test weeks and for each of the three XAI payloads (`none`, `shap`, `lime`), we:

1. **Select XAI values**
   - `none`: no dictionary, base = 0
   - `shap`: use `_shap_for(model, pos)`
   - `lime`: use `_lime_for(model, pos)`

2. **Construct prompt** via `make_prompt(...)` with:
   - `row_idx`: test‐row position
   - `model`: one of `"XGB"`, `"RandomForest"`, `"MLP"`
   - `perf`: performance metrics (MAE, RMSE, R²) from `metrics_df`
   - `features`: raw feature values for that week
   - `pred`: point forecast
   - `xai_kind`: `"none"`, `"shap"`, or `"lime"`
   - `xai_vals`, `base_val`: corresponding attribution values

3. **Store prompts**
   - In `HUMAN_PROMPTS_<MODEL>` keyed by `(row_pos, xai_kind)` for per‐model reuse.
   - In `HUMAN_PROMPTS_ALL` keyed by `(row_pos, model_name, xai_kind)` for global iteration.

4. **Print** each prompt with separators for manual review.

This yields nine prompts per model (3 rows × 3 XAI modes) and a total of 27 prompts in `HUMAN_PROMPTS_ALL`.


In [ ]:
# %% ────────────────────────────────────────────────────────────────────
# Prompts for **XGB**
# ───────────────────────────────────────────────────────────────────────
model_name = "XGB"
xai_modes  = ["none", "shap", "lime"]

HUMAN_PROMPTS_XGB = {}

for pos in row_sel:
    for xai in xai_modes:
        if xai == "shap":
            x_vals, base = _shap_for(model_name, pos)
        elif xai == "lime":
            x_vals, base = _lime_for(model_name, pos)
        else:                                 # none
            x_vals, base = {}, 0.0

        prompt = make_prompt(
            row_idx   = pos,
            model     = model_name,
            perf      = metrics_df.loc[model_name].to_dict(),
            features  = X_test.iloc[pos][FEATURES].to_dict(),
            pred      = preds[model_name][pos],
            xai_kind  = xai,
            xai_vals  = x_vals,
            base_val  = base,
        )

        HUMAN_PROMPTS_XGB[(pos, xai)] = prompt
        HUMAN_PROMPTS_ALL[(pos, model_name, xai)] = prompt

        print(f"\n{'='*80}\nRow {pos} | XGB | XAI {xai.upper()}\n{'-'*80}")
        print(prompt)


In [ ]:
# %% ────────────────────────────────────────────────────────────────────
# Prompts for **RandomForest**
# ───────────────────────────────────────────────────────────────────────
model_name = "RandomForest"
xai_modes  = ["none", "shap", "lime"]

HUMAN_PROMPTS_RF = {}

for pos in row_sel:
    for xai in xai_modes:
        if xai == "shap":
            x_vals, base = _shap_for(model_name, pos)
        elif xai == "lime":
            x_vals, base = _lime_for(model_name, pos)
        else:
            x_vals, base = {}, 0.0

        prompt = make_prompt(
            row_idx   = pos,
            model     = model_name,
            perf      = metrics_df.loc[model_name].to_dict(),
            features  = X_test.iloc[pos][FEATURES].to_dict(),
            pred      = preds[model_name][pos],
            xai_kind  = xai,
            xai_vals  = x_vals,
            base_val  = base,
        )

        HUMAN_PROMPTS_RF[(pos, xai)] = prompt
        HUMAN_PROMPTS_ALL[(pos, model_name, xai)] = prompt

        print(f"\n{'='*80}\nRow {pos} | RandomForest | XAI {xai.upper()}\n{'-'*80}")
        print(prompt)


In [ ]:
# %% ────────────────────────────────────────────────────────────────────
# Prompts for **MLP**
# ───────────────────────────────────────────────────────────────────────
model_name = "MLP"
xai_modes  = ["none", "shap", "lime"]

HUMAN_PROMPTS_MLP = {}

for pos in row_sel:
    for xai in xai_modes:
        if xai == "shap":
            x_vals, base = _shap_for(model_name, pos)
        elif xai == "lime":
            x_vals, base = _lime_for(model_name, pos)
        else:
            x_vals, base = {}, 0.0

        prompt = make_prompt(
            row_idx   = pos,
            model     = model_name,
            perf      = metrics_df.loc[model_name].to_dict(),
            features  = X_test.iloc[pos][FEATURES].to_dict(),
            pred      = preds[model_name][pos],
            xai_kind  = xai,
            xai_vals  = x_vals,
            base_val  = base,
        )

        HUMAN_PROMPTS_MLP[(pos, xai)] = prompt
        HUMAN_PROMPTS_ALL[(pos, model_name, xai)] = prompt

        print(f"\n{'='*80}\nRow {pos} | MLP | XAI {xai.upper()}\n{'-'*80}")
        print(prompt)


## 12 · Preparing SARIMAX input diagnostics

To elucidate the SARIMAX computation at each storyline week, we build a summary of its three core inputs:

1. **Full residual series**
   - Concatenate in‐sample residuals (`sarimax.resid` on training) and out‐of‐sample residuals from the rolling forecasts, yielding `SARIMAX_FULL_RESID`.

2. **Index mapping**
   - `ABS_IDX(i) = len(y_train) + i` converts a test‐set position `i` into the absolute index in both the full `weekly` data and `SARIMAX_FULL_RESID`.

3. **Input block function**
   - `_sarimax_inputs_block_with_eps(row_idx)` constructs:

     ```
     Model inputs for this step:
       • Last year’s observed value (yₜ₋₅₂): [True_Value at t–52]
       • Last week’s seasonally‐differenced value (Δ₅₂ yₜ₋₁): (y_t – yₜ₋₅₂)
       • Forecast error from 52 weeks ago (εₜ₋₅₂): [residual at t–52]
     ```

   - These inputs illustrate how the seasonal autoregressive term and seasonal‐MA error term combine to produce the new forecast.

This block ensures we can present the SARIMAX mechanics in the paper with exact numerical inputs for each example week.


## 13 · Generating SARIMAX‐specific prompts

For each selected test week, we build a tailored prompt that combines the generic forecasting context with detailed SARIMAX internals:

1. **Residual reconstruction**
   - Recompute `SARIMAX_FULL_RESID` by concatenating in‐sample and out‐of‐sample residuals.
   - `ABS_IDX(i)` maps a test‐row position to its absolute index in the residual series.

2. **Input diagnostics block** (`_sarimax_inputs_block_with_eps`)
   - Retrieves and formats:
     - **yₜ₋₅₂**: observed value 52 weeks earlier.
     - **Δ₅₂ yₜ₋₁**: seasonal difference from last week.
     - **εₜ₋₅₂**: forecast error from one year ago.

3. **Core prompt** (`make_prompt`) uses:
   - `model="SARIMAX (order=<auto_p,d,q>, seasonal_order=<auto_P,D,Q,52>)"`.
   - Performance metrics (MAE, RMSE, R²).
   - `Prediction` and the single exogenous feature `holiday_week_count`.
   - `xai_kind="none"` (no SHAP/LIME section).

4. **Appending SARIMAX details**
   - Insert the formatted inputs block.
   - Append the full `sarimax.summary()` text (model coefficients, diagnostics).
   - Add a note explaining that:
     > *“The SARIMAX order/seasonal_order were chosen via an `auto_arima` stepwise AICc search. Lagged demand effects are captured implicitly by the AR(1) and seasonal MA(52) terms; only `holiday_week_count` enters as an explicit regressor.”*

5. **Storage**
   - `HUMAN_PROMPTS_SARIMAX[pos]` for per‐model use.
   - `HUMAN_PROMPTS_ALL[(pos, "SARIMAX", "none")]` for global iteration.

> **Consistency check:** Ensure the SARIMAX model was actually fitted using these auto‐selected orders so that the prompt annotations match the underlying forecasts.


In [ ]:

# ----------------------------------------------------------------------
# 3️⃣  (Re)-build the prompt dicts
# ----------------------------------------------------------------------
sarimax_summary_txt = str(sarimax.summary())
HUMAN_PROMPTS_SARIMAX = {}

for pos in row_sel:
    prompt_core = make_prompt(
        row_idx   = pos,
        model     = f"SARIMAX (order={SARIMAX_ORDER}, seasonal_order={SARIMAX_SORDER})",
        perf      = metrics_df.loc["SARIMAX"].to_dict(),
        features  = {"holiday_week_count": float(X_test.iloc[pos]["holiday_week_count"])},
        pred      = preds["SARIMAX"][pos],
        xai_kind  = "none",
        xai_vals  = {},
        base_val  = 0.0,
    )

    prompt = (
        prompt_core + "\n\n" +
        sarimax_summary_txt + "\n\n"
        "[Note] The SARIMAX order/seasonal_order were chosen via an "
        "auto_arima stepwise AICc search. Lagged demand effects are captured "
        "implicitly by the AR and MA terms; only "
        "‘holiday_week_count’ enters as an explicit regressor."
    ).strip()

    HUMAN_PROMPTS_SARIMAX[pos] = prompt
    HUMAN_PROMPTS_ALL[(pos, "SARIMAX", "none")] = prompt

# quick peek
for pos, txt in HUMAN_PROMPTS_SARIMAX.items():
    print("="*78)
    print(f"Row {pos} | SARIMAX prompt \n" + "-"*78)
    print("\n".join(txt.splitlines()))

In [ ]:
# %% ─────────────────────────────────────────────────────────────────────
# 🔧  LLM helper toolbox  (credentials loaded from .env file)
# ────────────────────────────────────────────────────────────────────────
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# ╭──────────────────────────────────────────────────────────────────────╮
# │  1. Back-end configs (loaded from .env)                             │
# ╰──────────────────────────────────────────────────────────────────────╯
LLM_CONFIGS = {
    "GPT": {   # Azure OpenAI GPT-4o
        "provider": "azure",
        "azure_api_base":    os.getenv("GPT_AZURE_API_BASE"),
        "azure_api_version": os.getenv("GPT_AZURE_API_VERSION", "2023-12-01-preview"),
        "azure_api_key":     os.getenv("GPT_AZURE_API_KEY"),
        "deployment_name":   os.getenv("GPT_DEPLOYMENT_NAME", "gpt-4o"),
        "temperature":       float(os.getenv("GPT_TEMPERATURE", "1.0")),
    },
    "L3_LOCAL": {   # Ollama Llama-3
        "provider":   "openai_compat",
        "base_url":   os.getenv("L3_LOCAL_BASE_URL", "http://localhost:11434/v1"),
        "api_key":    os.getenv("L3_LOCAL_API_KEY", "ollama"),
        "model":      os.getenv("L3_LOCAL_MODEL", "llama3"),
        "temperature": float(os.getenv("L3_LOCAL_TEMPERATURE", "1.0")),
    },
    "DEEPSEEK": {   # Azure AI Studio DeepSeek-R1
        "provider":  "azureai",
        "endpoint":   os.getenv("DEEPSEEK_ENDPOINT"),
        "credential": os.getenv("DEEPSEEK_CREDENTIAL"),
        "model_name": os.getenv("DEEPSEEK_MODEL_NAME", "DeepSeek-R1-quevd"),
        "temperature": float(os.getenv("DEEPSEEK_TEMPERATURE", "1.0")),
    },
}

AVAILABLE_LLM_LABELS = list(LLM_CONFIGS.keys())  # → ['GPT', 'L3_LOCAL', 'DEEPSEEK']

# ╭──────────────────────────────────────────────────────────────────────╮
# │  2. Factory : get_llm(label, *, temperature=None)                   │
# ╰──────────────────────────────────────────────────────────────────────╯
def get_llm(label: str, *, temperature: float | None = None):
    """Return a chat-model instance for the given *label*.

    label must be one of AVAILABLE_LLM_LABELS.
    Optionally override temperature for ad-hoc sweeps.
    """
    if label not in LLM_CONFIGS:
        raise ValueError(f"Unknown label '{label}'. "
                         f"Choose from {AVAILABLE_LLM_LABELS}")
    cfg  = LLM_CONFIGS[label]
    prov = cfg["provider"]
    temp = temperature if temperature is not None else cfg.get("temperature", 1.0)

    if prov == "azure":          # Azure OpenAI
        return AzureChatOpenAI(
            openai_api_base    = cfg["azure_api_base"],
            openai_api_version = cfg["azure_api_version"],
            openai_api_key     = cfg["azure_api_key"],
            deployment_name    = cfg["deployment_name"],
            temperature        = temp,
        )

    if prov == "openai_compat":  # Ollama
        return ChatOpenAI(
            base_url    = cfg["base_url"],
            api_key     = cfg["api_key"],
            model       = cfg["model"],
            temperature = temp,
        )

    if prov == "azureai":        # Azure AI Studio (DeepSeek)
        client = AzureAIChatCompletionsModel(
            endpoint    = cfg["endpoint"],
            credential  = cfg["credential"],
            model_name  = cfg["model_name"],
            temperature = temp,
        )
        return ChatAzureAIInference(client=client)

    raise ValueError(f"Unknown provider '{prov}' for label '{label}'")

In [ ]:
system_prompt_template = """
Interpret the time-series forecasting context that follows.

Your goal is to help a non-technical lay user. The user does not have a background in statistical models, machine learning, times-series or explainability methods (e.g. SHAP,LIME). The user needs to understand
• **why** the model produced this forecast, and
• **how much confidence** can be placed in it.


OUTPUT RULES
• Write **up to six bullet points** — no more.
• Keep the whole response **≤ 200 words**.
• Use plain language, but it’s fine to employ key time-series terms such as *lag*, *trend*,
  *seasonality*, *baseline*, *error*.
• Do not reveal code.
"""

In [ ]:
# %% ─────────────────────────────────────────────────────────────────────
# 📦  Shared helpers for collecting & saving explanation rows
#      (place once; reuse from every strategy cell)
# -----------------------------------------------------------------------

# Folder …/outputs_final/explanations
_SAVE_ROOT = Path("outputs_final") / "explanations"
_SAVE_ROOT.mkdir(parents=True, exist_ok=True)

# In-memory buffer  ⟶  {strategy_name: [row_dict, …]}
_ROWS_BUFFER: dict[str, list[dict]] = {}


def _json_or_none(obj):
    """Dict / list → pretty JSON; empties → '<none>'."""
    if obj in (None, {}, []):
        return "<none>"
    return _json.dumps(obj, ensure_ascii=False)


def add_row(strategy: str, row: dict):
    """
    Append one row (dict) for *strategy*.
    Null / NaN → '<none>'; dict-like cols dumped to JSON strings.
    """
    norm = {}
    for k, v in row.items():
        if v is None or (isinstance(v, float) and _pd.isna(v)):
            norm[k] = "<none>"
        else:
            norm[k] = v

    # JSON-encode potential dict columns
    for col in ("Feature_vals", "XAI_vals"):
        norm[col] = _json_or_none(norm.get(col))

    _ROWS_BUFFER.setdefault(strategy, []).append(norm)


def save_csv(strategy: str):
    """
    Flush buffer to CSV  →  outputs_final/explanations/<strategy>.csv
    """
    if strategy not in _ROWS_BUFFER:
        print(f"[save_csv] nothing buffered for strategy='{strategy}'")
        return
    df  = _pd.DataFrame(_ROWS_BUFFER[strategy])
    out = _SAVE_ROOT / f"{strategy}.csv"
    df.to_csv(out, index=False)
    print(f"✅  Saved {len(df):,} rows → {out}")


In [ ]:
# %% ─────────────────────────────────────────────────────────────────────
# ⚙️  MODE SWITCH  –  'test'  vs  'prod'
#     • In *test* mode: trims LLM list & keeps 1 prompt/model (see below)
# -----------------------------------------------------------------------

if RUN_MODE == "test":
    print("🚧  TEST MODE ENABLED — performing fast-run filters\n")

    # ── 1) Drop Ollama / Llama-3 backend ───────────────────────────────
    if "L3_LOCAL" in AVAILABLE_LLM_LABELS:
        AVAILABLE_LLM_LABELS.remove("L3_LOCAL")
        print("   ❎  Removed 'L3_LOCAL' from AVAILABLE_LLM_LABELS")
    print("   ✅  Remaining LLMs:", AVAILABLE_LLM_LABELS, "\n")

    # ── 2) Retain exactly ONE prompt per model ─────────────────────────
    _test_row = row_sel[0]   # first storyline position
    _kept     = {}

    for (row_pos, model_name, xai_kind), prompt_txt in HUMAN_PROMPTS_ALL.items():
        if row_pos != _test_row:
            continue

        # ML models → keep LIME explanation
        if model_name != "SARIMAX" and xai_kind == "lime":
            _kept[(row_pos, model_name, xai_kind)] = prompt_txt

        # SARIMAX  → keep plain (none) explanation
        if model_name == "SARIMAX" and xai_kind == "none":
            _kept[(row_pos, model_name, xai_kind)] = prompt_txt

    HUMAN_PROMPTS_ALL = _kept
    print(f"   📄  Kept {len(HUMAN_PROMPTS_ALL)} prompts:")
    for key in HUMAN_PROMPTS_ALL:
        print("      ", key)

else:
    print("🟢  PRODUCTION MODE — no filters applied")


In [ ]:
# %% ─────────────────────────────────────────────────────────────────────
# 🚀  ZERO-SHOT  – generates explanations & writes zero_shot.csv
#      (relies on: system_prompt_template, HUMAN_PROMPTS_ALL,
#       FEATURES, SHAP_INFO, lime_full_df, preds, test, metrics_df,
#       AVAILABLE_LLM_LABELS, get_llm, call_with_retry, add_row, save_csv)
# -----------------------------------------------------------------------

STRATEGY_NAME = "zero_shot"
THINK_RE = re.compile(r"<think>(.*?)</think>", re.I | re.S)

for llm_label in AVAILABLE_LLM_LABELS:
    print(f"\n### ZERO-SHOT | LLM = {llm_label}")
    llm = get_llm(llm_label)

    for (row_pos, model_name, xai_kind), human_msg in HUMAN_PROMPTS_ALL.items():

        # ── call model ────────────────────────────────────────────────
        msgs = [
            SystemMessage(content=system_prompt_template.strip()),
            HumanMessage(content=human_msg),
        ]
        t0 = time.time()
        resp = call_with_retry(llm, msgs)
        dt  = time.time() - t0
        raw = resp.content.strip()

        # ── split <think> … </think>  (if present) --------------------
        m = THINK_RE.match(raw)
        if m:
            internal = m.group(1).strip()
            answer   = raw[m.end():].strip()
        else:
            internal = "<none>"
            answer   = raw

        # ── core numbers & meta --------------------------------------
        true_val = float(y_test.iloc[row_pos])
        pred_val = float(preds[model_name][row_pos])
        pct_err  = (pred_val - true_val) / true_val * 100

        feat_vals = test.iloc[row_pos][FEATURES].to_dict()

        xai_vals, base_val = {}, "<none>"
        if xai_kind == "shap":
            vec       = SHAP_INFO[model_name]["values"][row_pos]
            xai_vals  = dict(zip(FEATURES, map(float, vec)))
            base_val  = float(SHAP_INFO[model_name]["base_value"])
        elif xai_kind == "lime":
            row_key   = X_test.index[row_pos]          # label stored in LIME df
            row_lime  = lime_full_df.loc[(model_name, row_key)]
            base_val  = float(row_lime["intercept"])
            xai_vals  = row_lime.drop("intercept").to_dict()

        add_row(STRATEGY_NAME, dict(
            LLM            = llm_label,
            Temperature    = "<default>",
            SampleID       = 1,
            RowPos         = row_pos,
            Model          = model_name,
            XAI            = xai_kind,
            WeekEndDate    = test.iloc[row_pos]["DateTime"],
            Prediction     = pred_val,
            TrueValue      = true_val,
            PctError       = pct_err,
            Duration_s     = dt,
            TokensThought  = n_tokens(internal),
            TokensAnswer   = n_tokens(answer),
            TokensTotal    = n_tokens(raw),
            MAE            = metrics_df.at[model_name, "MAE"],
            RMSE           = metrics_df.at[model_name, "RMSE"],
            R2             = metrics_df.at[model_name, "R2"],
            SystemMsg      = system_prompt_template.strip(),
            HumanMsg       = human_msg,
            InternalThought= internal,
            Explanation    = answer,
            XAI_base_value = base_val,
            Feature_vals   = feat_vals,
            XAI_vals       = xai_vals,
        ))

        tok_tot = n_tokens(raw)
        print(f"  ✔ Row {row_pos:3d} | {model_name:<11} | {xai_kind:<4} | "
              f"{dt:5.2f}s | tok≈{tok_tot}")

# finally write CSV
save_csv(STRATEGY_NAME)


In [ ]:
# %% ─────────────────────────────────────────────────────────────────────
# 📄  ZERO-SHOT  •  TEMPERATURE SWEEP  •  SAVE AS CSV
#     strategy name = "zero_shot_temp"
# -----------------------------------------------------------------------
STRATEGY_NAME = "zero_shot_temp"

# ─ sweep settings ──────────────────────────────────────────────────────
MIN_TEMP, MAX_TEMP, TEMP_STEP = 0.0, 1.4, 0.2
N_SAMPLES = 3
temps = [round(MIN_TEMP + i*TEMP_STEP, 2)
         for i in range(int((MAX_TEMP-MIN_TEMP)/TEMP_STEP) + 1)]

SYSTEM_MSG = SystemMessage(content=system_prompt_template.strip())
THINK_RE   = re.compile(r"<think>(.*?)</think>", re.I | re.S)

for llm_label in AVAILABLE_LLM_LABELS:
    for T in temps:
        llm_T = get_llm_temp(llm_label, T)
        print(f"\n### TEMP-SWEEP | {llm_label} | T={T:.1f}")

        for sample_id in range(1, N_SAMPLES+1):
            print(f"  … sample {sample_id}")

            for (row_pos, model_name, xai_kind), human_txt in HUMAN_PROMPTS_ALL.items():

                # ----- LLM call --------------------------------------------------
                msgs = [SYSTEM_MSG, HumanMessage(content=human_txt)]
                t0   = time.time()
                resp = call_with_retry(llm_T, msgs)
                dt_s = time.time() - t0
                raw  = resp.content.strip()

                # ----- split potential <think> block ----------------------------
                m = THINK_RE.match(raw)
                if m:
                    internal = m.group(1).strip()
                    answer   = raw[m.end():].strip()
                else:
                    internal = "<none>"
                    answer   = raw

                # ----- metrics & context ----------------------------------------
                true_v  = float(y_test.iloc[row_pos])
                pred_v  = float(preds[model_name][row_pos])
                pct_err = (pred_v - true_v) / true_v * 100
                feat_v  = test.iloc[row_pos][FEATURES].to_dict()

                xai_vals, base_val = {}, None
                if xai_kind == "shap":
                    vec      = SHAP_INFO[model_name]["values"][row_pos]
                    xai_vals = dict(zip(FEATURES, map(float, vec)))
                    base_val = float(SHAP_INFO[model_name]["base_value"])
                elif xai_kind == "lime":
                    lime_row = lime_full_df.loc[(model_name, test.index[row_pos])]
                    base_val = float(lime_row["intercept"])
                    xai_vals = lime_row.drop("intercept").to_dict()

                # ----- add row ---------------------------------------------------
                add_row(STRATEGY_NAME, dict(
                    LLM            = llm_label,
                    Temperature    = T,
                    SampleID       = sample_id,
                    RowPos         = row_pos,
                    Model          = model_name,
                    XAI            = xai_kind,
                    WeekEndDate    = test.iloc[row_pos]["DateTime"],
                    Prediction     = pred_v,
                    TrueValue      = true_v,
                    PctError       = pct_err,
                    Duration_s     = dt_s,
                    TokensThought  = n_tokens(internal),
                    TokensAnswer   = n_tokens(answer),
                    TokensTotal    = n_tokens(raw),
                    MAE            = metrics_df.at[model_name, "MAE"],
                    RMSE           = metrics_df.at[model_name, "RMSE"],
                    R2             = metrics_df.at[model_name, "R2"],
                    SystemMsg      = SYSTEM_MSG.content,
                    HumanMsg       = human_txt,
                    InternalThought= internal,
                    Explanation    = answer,
                    XAI_base_value = base_val,
                    Feature_vals   = feat_v,
                    XAI_vals       = xai_vals,
                ))

                print(f"    ✔ Row {row_pos:3d} | {model_name:<11} | "
                      f"{xai_kind:<4} | {dt_s:5.2f}s | tok≈{n_tokens(raw)}")

# flush to CSV
save_csv(STRATEGY_NAME)
print(f"✅  Saved → explanations/{STRATEGY_NAME}.csv")

In [ ]:
# %% ─────────────────────────────────────────────────────────────────────
# 📝  FEW-SHOT STRATEGY (2 in-context examples) → explanations/few_shot.csv
# -----------------------------------------------------------------------
STRATEGY_NAME = "few_shot"

SYSTEM_MSG = SystemMessage(content=system_prompt_template.strip())

# full in-context examples (same as before) -----------------------------
fewshot_h1 = HumanMessage(content="""\
The following is about time-series data with a single-step-ahead prediction, where the model predicts the next value in the time series based on previous observations.
Data Domain: Retail Sales
Dataset Description:
- The dataset contains 469 weekly sales totals from a mid-size supermarket chain in Madrid, Jan 2015 – Dec 2023.
- Originally daily cash-register data, aggregated to weekly sums.
- Lag_1…Lag_8, ISO week number, number of national holidays per week, and Promotion_Intensity (0–1) were added as features.
- Target: weekly gross sales (thousand €).

Model Used: XGBRegressor
Model Performance:
  - MAE: 12.300
  - RMSE: 16.800
  - R²: 0.874

Prediction: 242.70

Instance Features or Context:
lag_1: 238.4
lag_2: 245.1
lag_3: 230.9
lag_4: 225.6
lag_5: 220.3
lag_6: 215.0
lag_7: 210.8
lag_8: 205.7
weekofyear: 48
holiday_week_count: 1
promotion_intensity: 0.65

SHAP values:
promotion_intensity: 6.900
lag_1: 4.200
holiday_week_count: 3.500
lag_2: -2.800
lag_3: -1.100
lag_4: -0.700
weekofyear: 0.400
lag_5: -0.200
lag_6: -0.100
lag_7: 0.000
lag_8: 0.000
The expected/base value for SHAP: 235.100
""")

fewshot_a1 = AIMessage(content="""\
• SHAP starts from a baseline of ≈ 235 k€ and shows how each feature nudges the forecast up (positive) or down (negative).
• Promotion intensity (0.65) is the main driver, adding ≈ 7 k€.
• Last-week sales (lag 1 = 238 k€) add ≈ 4 k€, showing continued momentum.
• One public holiday lifts demand by ≈ 3.5 k€, but the boost is modest.
• Week 48 sits before Christmas; the tiny +0.4 k€ seasonal bump hints demand hasn’t peaked yet.
• Negative pulls from lags 2-4 temper the forecast; 242.7 k€ is plausible within the ±16.8 k€ RMSE.
""")

# ── 3. FULL EXAMPLE #2  (traffic volume, LIME) ─────────────────────────
fewshot_h2 = HumanMessage(content="""\
The following is about time series data with a single-step ahead prediction, where the model predicts the next value in the time series based on previous observations.
Data Domain: Traffic Volume
Dataset Description:
- The dataset contains 3,000,000 hourly vehicle counts from a freeway loop detector in Seattle, Jan 2018 – May 2025.
- Weather records joined, series kept at hourly resolution.
- Lag_1…Lag_6, Hour_of_Day, Is_Weekend, and Precipitation_mm were added as features.
- Target: vehicle count per hour.

Model Used: LSTM
Model Performance:
  - MAE: 22.800
  - RMSE: 29.500
  - R²: 0.640

Prediction: 621

Instance Features or Context:
lag_1: 598
lag_2: 610
lag_3: 630
lag_4: 645
lag_5: 660
lag_6: 672
hour_of_day: 18
is_weekend: 0
precipitation_mm: 0.0

LIME values:
lag_1: 8.700
hour_of_day: 5.400
lag_4: -4.800
precipitation_mm: -3.900
lag_2: 3.200
lag_3: -2.600
is_weekend: 1.100
lag_5: -1.000
lag_6: 0.400
The expected/base value for LIME: 600.500
""")

fewshot_a2 = AIMessage(content="""\
• LIME builds a local surrogate: starting from ≈ 600 vehicles, each feature nudges the prediction.
• lag 1 (= 598) adds +8.7 → slight rebound after a dip.
• Rush-hour flag (hour 18) adds +5.4.
• lag 4 (= 645) subtracts −4.8, hinting the earlier surge is fading.
• Clear weather (−3.9) offsets the “weekday” +1.1 effect.
• Net +20 over baseline → 621, within ±29.5 RMSE, so forecast looks reasonable.
""")


fewshot_h3 = HumanMessage(content="""\
The following is about time series data with a single-step ahead prediction.
Data Domain: Cryptocurrency Pricing
Dataset Description:
- Contains 730 daily closing prices for Dogecoin from Jan 2021-Dec 2022
- Features: lag_1...lag_5, day_of_week, social_media_mentions
- Target: next day's price change (USD)

Model Used: NaiveBaseline
Model Performance:
  - MAE: 0.482
  - RMSE: 0.721
  - R²: 0.086

Prediction: +0.35 USD

Instance Features or Context:
lag_1: -0.12
lag_2: +0.51
lag_3: -0.08
lag_4: +0.22
lag_5: -0.31
day_of_week: 3 (Wednesday)
social_media_mentions: 4821

SHAP values:
social_media_mentions: 0.210
lag_2: 0.185
day_of_week: -0.032
lag_4: 0.021
lag_5: -0.018
lag_1: 0.012
lag_3: -0.005
The expected/base value for SHAP: 0.092""")

fewshot_a3 = AIMessage(content="""\
• This model has poor accuracy (MAE=$0.48, R²=0.086) - its predictions should not be trusted for financial decisions.
• Despite the +$0.35 forecast, the high error means actual results could easily be -$0.13 to +$0.83.
• Social media mentions (4821) are the strongest positive driver but cryptocurrency markets are highly volatile.
• The weak relationship to historical patterns (low lag contributions) indicates fundamental unpredictability.
• With such low explanatory power (R²<0.1), this forecast is essentially an educated guess.
• Never rely on low-performance models for high-stakes decisions like investments.""")

fewshot_h4 = HumanMessage(content="""\
The following is about time series data with a single-step ahead prediction, where the model predicts the next value in the time series based on previous observations.
Data Domain: Airline Passenger Volume
Dataset Description:
- Contains 144 monthly passenger counts (thousands) for an airline company (2010-2022)
- Features: month_number, holiday_flag
- Target: monthly passengers

Model Used: SARIMAX(order=(1,1,1), seasonal_order=(1,1,1,12))
Model Performance:
  - MAE: 12.8
  - RMSE: 16.4
  - R²: 0.942

Prediction: 412.3 k passengers

Instance Features or Context:
holiday_flag: 1 (public-holiday month)

SARIMAX Results:
=========================================================================================
Dep. Variable:                        passengers   No. Observations:                  132
Model:             SARIMAX(1,1,1)x(1,1,1,12)      Log Likelihood                -402.771
Date:                            Fri, 15 Dec 2023   AIC                            815.542
Time:                                    09:30:15   BIC                            830.123
Sample:                                01-01-2010   HQIC                           821.456
                                     − 12-01-2020
Covariance Type:                              opg
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.3184      0.092      3.456      0.001       0.138       0.499
ma.L1         -0.2297      0.118     -1.952      0.051      -0.460       0.001
ar.S.L12       0.6921      0.076      9.132      0.000       0.544       0.841
ma.S.L12      -0.3815      0.094     -4.049      0.000      -0.566      -0.197
holiday_flag  -8.7721      3.211     -2.732      0.006     -15.066      -2.478
sigma2        89.2513     12.633      7.065      0.000      64.531     113.972
===================================================================================
Ljung-Box (L1) (Q):                   0.24   Jarque-Bera (JB):                 0.85
Prob(Q):                              0.62   Prob(JB):                         0.65
Heteroskedasticity (H):               1.07   Skew:                            −0.12
Prob(H) (two-sided):                  0.82   Kurtosis:                         2.91
===================================================================================

[Note] The SARIMAX order/seasonal_order were chosen via an auto_arima stepwise AICc search. Lagged effects are captured implicitly by the AR and MA terms; only ‘holiday_flag’ enters as an explicit regressor.
""")

fewshot_a4 = AIMessage(content="""\
• **Seasonal heartbeat** – About 69 % of today’s passengers come from the *same month last year* (coefficient 0.69), so the model leans heavily on the predictable yearly rhythm of air travel.
• **Month-to-month carry-over** – A smaller slice (32 %) comes from *last month* (AR 0.32), smoothing sudden spikes or dips rather than copying them outright.
• **Holiday adjustment** – If the month has a public holiday, the model subtracts roughly **8.8 k passengers** because leisure demand never fully replaces missing business trips.
• **Error “shock absorbers”** – The two negative MA numbers (-0.23 short-term, -0.38 seasonal) nudge forecasts down after an over-shoot and up after an under-shoot, so mistakes don’t snowball.
• **How good is it?** – With **R² = 0.94** and a typical miss of **±12.8 k** passengers (≈ 3 %), the model tracks history closely.
• **What to expect** – Given those errors and the random-looking residual checks, the **412 k-passenger** forecast is likely within **±17 k** unless unusual events disrupt the usual pattern.
""")

fewshot_msgs = [fewshot_h1, fewshot_a1, fewshot_h2, fewshot_a2, fewshot_h3, fewshot_a3, fewshot_h4, fewshot_a4]

THINK_RE = re.compile(r"<think>(.*?)</think>", re.I | re.S)

for llm_label in AVAILABLE_LLM_LABELS:
    llm = get_llm(llm_label)
    print(f"\n### FEW-SHOT | LLM = {llm_label}")

    for (row_pos, model_name, xai_kind), human_txt in HUMAN_PROMPTS_ALL.items():

        msgs = [SYSTEM_MSG, *fewshot_msgs, HumanMessage(content=human_txt)]

        t0   = time.time()
        resp = call_with_retry(llm, msgs)
        dt_s = time.time() - t0
        raw  = resp.content.strip()

        # — split optional <think> block --------------------------------
        m = THINK_RE.match(raw)
        if m:
            internal = m.group(1).strip()
            answer   = raw[m.end():].strip()
        else:
            internal = "<none>"
            answer   = raw

        # — core numbers ------------------------------------------------
        true_v  = float(y_test.iloc[row_pos])
        pred_v  = float(preds[model_name][row_pos])
        pct_err = (pred_v - true_v) / true_v * 100
        feat_v  = test.iloc[row_pos][FEATURES].to_dict()

        xai_vals, base_val = {}, None
        if xai_kind == "shap":
            vec      = SHAP_INFO[model_name]["values"][row_pos]
            xai_vals = dict(zip(FEATURES, map(float, vec)))
            base_val = float(SHAP_INFO[model_name]["base_value"])
        elif xai_kind == "lime":
            lime_row = lime_full_df.loc[(model_name, test.index[row_pos])]
            base_val = float(lime_row["intercept"])
            xai_vals = lime_row.drop("intercept").to_dict()

        # — add row to buffer -------------------------------------------
        add_row(STRATEGY_NAME, dict(
            LLM            = llm_label,
            Temperature    = "<default>",
            SampleID       = 1,
            RowPos         = row_pos,
            Model          = model_name,
            XAI            = xai_kind,
            WeekEndDate    = test.iloc[row_pos]["DateTime"],
            Prediction     = pred_v,
            TrueValue      = true_v,
            PctError       = pct_err,
            Duration_s     = dt_s,
            TokensThought  = n_tokens(internal),
            TokensAnswer   = n_tokens(answer),
            TokensTotal    = n_tokens(raw),
            MAE            = metrics_df.at[model_name, "MAE"],
            RMSE           = metrics_df.at[model_name, "RMSE"],
            R2             = metrics_df.at[model_name, "R2"],
            SystemMsg      = SYSTEM_MSG.content,
            HumanMsg       = human_txt,
            InternalThought= internal,
            Explanation    = answer,
            XAI_base_value = base_val,
            Feature_vals   = feat_v,
            XAI_vals       = xai_vals,
        ))

        print(f"  ✔ Row {row_pos:3d} | {model_name:<11} | {xai_kind:<4}"
              f" | {dt_s:5.2f}s | tok≈{n_tokens(raw)}")

# flush to CSV
save_csv(STRATEGY_NAME)
print(f"✅  Saved → explanations/{STRATEGY_NAME}.csv")


In [ ]:
# %% ─────────────────────────────────────────────────────────────────────
# 🧠  CHAIN-OF-THOUGHT  •  ZERO-SHOT
#     strategy name = "cot_zero_shot"
#     – adds “Think step-by-step.” to the system prompt
#     – SKIPS DeepSeek (per user request)
# -----------------------------------------------------------------------
STRATEGY_NAME = "cot_zero_shot"
THINK_TAG_RE  = re.compile(r"<think>(.*?)</think>", re.I | re.S)

for llm_label in AVAILABLE_LLM_LABELS:
    if llm_label == "DEEPSEEK":
        print(f"⏭️  Skipping CoT for {llm_label}")
        continue

    llm = get_llm(llm_label)
    print(f"\n### COT ZERO-SHOT | LLM = {llm_label}")

    # system message with CoT cue
    sys_msg = SystemMessage(content=system_prompt_template.strip() + "\n\nThink step-by-step.")

    for (row_pos, model_name, xai_kind), human_txt in HUMAN_PROMPTS_ALL.items():

        # ── call the model ────────────────────────────────────────────
        msgs = [sys_msg, HumanMessage(content=human_txt)]
        t0   = time.time()
        resp = call_with_retry(llm, msgs)
        dt_s = time.time() - t0
        raw  = resp.content.strip()

        # ── optional <think> … </think> split (works for GPT / Ollama) ─
        m = THINK_TAG_RE.match(raw)
        internal = m.group(1).strip() if m else "<none>"
        answer   = raw[m.end():].strip() if m else raw

        # ── gather numbers / meta info  -------------------------------
        true_val = float(y_test.iloc[row_pos])
        pred_val = float(preds[model_name][row_pos])
        pct_err  = (pred_val - true_val) / true_val * 100
        feat_vals = test.iloc[row_pos][FEATURES].to_dict()

        # --- XAI payload ---------------------------------------------
        xai_vals, base_val = {}, None
        if xai_kind == "shap":
            vec       = SHAP_INFO[model_name]["values"][row_pos]
            xai_vals  = dict(zip(FEATURES, map(float, vec)))
            base_val  = float(SHAP_INFO[model_name]["base_value"])
        elif xai_kind == "lime":
            row_lime  = lime_full_df.loc[(model_name, test.index[row_pos])]
            base_val  = float(row_lime["intercept"])
            xai_vals  = row_lime.drop("intercept").to_dict()

        # --- save via helper -----------------------------------------
        add_row(STRATEGY_NAME, dict(
            LLM            = llm_label,
            Temperature    = "<default>",
            SampleID       = 1,
            RowPos         = row_pos,
            Model          = model_name,
            XAI            = xai_kind,
            WeekEndDate    = test.iloc[row_pos]["DateTime"],
            Prediction     = pred_val,
            TrueValue      = true_val,
            PctError       = pct_err,
            Duration_s     = dt_s,
            TokensThought  = n_tokens(internal),
            TokensAnswer   = n_tokens(answer),
            TokensTotal    = n_tokens(raw),
            MAE            = metrics_df.at[model_name, "MAE"],
            RMSE           = metrics_df.at[model_name, "RMSE"],
            R2             = metrics_df.at[model_name, "R2"],
            SystemMsg      = sys_msg.content,
            HumanMsg       = human_txt,
            InternalThought= internal,
            Explanation    = answer,
            XAI_base_value = base_val,
            Feature_vals   = feat_vals,
            XAI_vals       = xai_vals,
        ))

        print(f"  ✔ Row {row_pos:3d} | {model_name:<11} | {xai_kind:<4} | "
              f"{dt_s:5.2f}s | tok≈{n_tokens(raw)}")

# flush to disk
save_csv(STRATEGY_NAME)


In [ ]:
# %% ─────────────────────────────────────────────────────────────────────
# 🧠  CHAIN-OF-THOUGHT  •  FEW-SHOT  →  explanations/cot_few_shot.csv
#     – Adds “Think step-by-step.” to the system prompt
#     – Re-uses the two context examples (SHAP & LIME)
#     – SKIPS DeepSeek  (per user request)
# -----------------------------------------------------------------------
STRATEGY_NAME = "cot_few_shot"
THINK_TAG_RE  = re.compile(r"<think>(.*?)</think>", re.I | re.S)

# ── system message with CoT cue ─────────────────────────────────────────
SYSTEM_MSG = SystemMessage(content=system_prompt_template.strip()
                           + "\n\nThink step-by-step.")

# full in-context examples (same as before) -----------------------------
fewshot_h1 = HumanMessage(content="""\
The following is about time-series data with a single-step-ahead prediction, where the model predicts the next value in the time series based on previous observations.
Data Domain: Retail Sales
Dataset Description:
- The dataset contains 469 weekly sales totals from a mid-size supermarket chain in Madrid, Jan 2015 – Dec 2023.
- Originally daily cash-register data, aggregated to weekly sums.
- Lag_1…Lag_8, ISO week number, number of national holidays per week, and Promotion_Intensity (0–1) were added as features.
- Target: weekly gross sales (thousand €).

Model Used: XGBRegressor
Model Performance:
  - MAE: 12.300
  - RMSE: 16.800
  - R²: 0.874

Prediction: 242.70

Instance Features or Context:
lag_1: 238.4
lag_2: 245.1
lag_3: 230.9
lag_4: 225.6
lag_5: 220.3
lag_6: 215.0
lag_7: 210.8
lag_8: 205.7
weekofyear: 48
holiday_week_count: 1
promotion_intensity: 0.65

SHAP values:
promotion_intensity: 6.900
lag_1: 4.200
holiday_week_count: 3.500
lag_2: -2.800
lag_3: -1.100
lag_4: -0.700
weekofyear: 0.400
lag_5: -0.200
lag_6: -0.100
lag_7: 0.000
lag_8: 0.000
The expected/base value for SHAP: 235.100
""")

fewshot_a1 = AIMessage(content="""\
• SHAP starts from a baseline of ≈ 235 k€ and shows how each feature nudges the forecast up (positive) or down (negative).
• Promotion intensity (0.65) is the main driver, adding ≈ 7 k€.
• Last-week sales (lag 1 = 238 k€) add ≈ 4 k€, showing continued momentum.
• One public holiday lifts demand by ≈ 3.5 k€, but the boost is modest.
• Week 48 sits before Christmas; the tiny +0.4 k€ seasonal bump hints demand hasn’t peaked yet.
• Negative pulls from lags 2-4 temper the forecast; 242.7 k€ is plausible within the ±16.8 k€ RMSE.
""")

# ── 3. FULL EXAMPLE #2  (traffic volume, LIME) ─────────────────────────
fewshot_h2 = HumanMessage(content="""\
The following is about time series data with a single-step ahead prediction, where the model predicts the next value in the time series based on previous observations.
Data Domain: Traffic Volume
Dataset Description:
- The dataset contains 3,000,000 hourly vehicle counts from a freeway loop detector in Seattle, Jan 2018 – May 2025.
- Weather records joined, series kept at hourly resolution.
- Lag_1…Lag_6, Hour_of_Day, Is_Weekend, and Precipitation_mm were added as features.
- Target: vehicle count per hour.

Model Used: LSTM
Model Performance:
  - MAE: 22.800
  - RMSE: 29.500
  - R²: 0.640

Prediction: 621

Instance Features or Context:
lag_1: 598
lag_2: 610
lag_3: 630
lag_4: 645
lag_5: 660
lag_6: 672
hour_of_day: 18
is_weekend: 0
precipitation_mm: 0.0

LIME values:
lag_1: 8.700
hour_of_day: 5.400
lag_4: -4.800
precipitation_mm: -3.900
lag_2: 3.200
lag_3: -2.600
is_weekend: 1.100
lag_5: -1.000
lag_6: 0.400
The expected/base value for LIME: 600.500
""")

fewshot_a2 = AIMessage(content="""\
• LIME builds a local surrogate: starting from ≈ 600 vehicles, each feature nudges the prediction.
• lag 1 (= 598) adds +8.7 → slight rebound after a dip.
• Rush-hour flag (hour 18) adds +5.4.
• lag 4 (= 645) subtracts −4.8, hinting the earlier surge is fading.
• Clear weather (−3.9) offsets the “weekday” +1.1 effect.
• Net +20 over baseline → 621, within ±29.5 RMSE, so forecast looks reasonable.
""")

fewshot_h3 = HumanMessage(content="""\
The following is about time series data with a single-step ahead prediction.
Data Domain: Cryptocurrency Pricing
Dataset Description:
- Contains 730 daily closing prices for Dogecoin from Jan 2021-Dec 2022
- Features: lag_1...lag_5, day_of_week, social_media_mentions
- Target: next day's price change (USD)

Model Used: NaiveBaseline
Model Performance:
  - MAE: 0.482
  - RMSE: 0.721
  - R²: 0.086

Prediction: +0.35 USD

Instance Features or Context:
lag_1: -0.12
lag_2: +0.51
lag_3: -0.08
lag_4: +0.22
lag_5: -0.31
day_of_week: 3 (Wednesday)
social_media_mentions: 4821

SHAP values:
social_media_mentions: 0.210
lag_2: 0.185
day_of_week: -0.032
lag_4: 0.021
lag_5: -0.018
lag_1: 0.012
lag_3: -0.005
The expected/base value for SHAP: 0.092""")

fewshot_a3 = AIMessage(content="""\
• This model has poor accuracy (MAE=$0.48, R²=0.086) - its predictions should not be trusted for financial decisions.
• Despite the +$0.35 forecast, the high error means actual results could easily be -$0.13 to +$0.83.
• Social media mentions (4821) are the strongest positive driver but cryptocurrency markets are highly volatile.
• The weak relationship to historical patterns (low lag contributions) indicates fundamental unpredictability.
• With such low explanatory power (R²<0.1), this forecast is essentially an educated guess.
• Never rely on low-performance models for high-stakes decisions like investments.""")

fewshot_h4 = HumanMessage(content="""\
The following is about time series data with a single-step ahead prediction, where the model predicts the next value in the time series based on previous observations.
Data Domain: Airline Passenger Volume
Dataset Description:
- Contains 144 monthly passenger counts (thousands) for an airline company (2010-2022)
- Features: month_number, holiday_flag
- Target: monthly passengers

Model Used: SARIMAX(order=(1,1,1), seasonal_order=(1,1,1,12))
Model Performance:
  - MAE: 12.8
  - RMSE: 16.4
  - R²: 0.942

Prediction: 412.3 k passengers

Instance Features or Context:
holiday_flag: 1 (public-holiday month)

SARIMAX Results:
=========================================================================================
Dep. Variable:                        passengers   No. Observations:                  132
Model:             SARIMAX(1,1,1)x(1,1,1,12)      Log Likelihood                -402.771
Date:                            Fri, 15 Dec 2023   AIC                            815.542
Time:                                    09:30:15   BIC                            830.123
Sample:                                01-01-2010   HQIC                           821.456
                                     − 12-01-2020
Covariance Type:                              opg
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.3184      0.092      3.456      0.001       0.138       0.499
ma.L1         -0.2297      0.118     -1.952      0.051      -0.460       0.001
ar.S.L12       0.6921      0.076      9.132      0.000       0.544       0.841
ma.S.L12      -0.3815      0.094     -4.049      0.000      -0.566      -0.197
holiday_flag  -8.7721      3.211     -2.732      0.006     -15.066      -2.478
sigma2        89.2513     12.633      7.065      0.000      64.531     113.972
===================================================================================
Ljung-Box (L1) (Q):                   0.24   Jarque-Bera (JB):                 0.85
Prob(Q):                              0.62   Prob(JB):                         0.65
Heteroskedasticity (H):               1.07   Skew:                            −0.12
Prob(H) (two-sided):                  0.82   Kurtosis:                         2.91
===================================================================================

[Note] The SARIMAX order/seasonal_order were chosen via an auto_arima stepwise AICc search. Lagged effects are captured implicitly by the AR and MA terms; only ‘holiday_flag’ enters as an explicit regressor.
""")

fewshot_a4 = AIMessage(content="""\
• **Seasonal heartbeat** – About 69 % of today’s passengers come from the *same month last year* (coefficient 0.69), so the model leans heavily on the predictable yearly rhythm of air travel.
• **Month-to-month carry-over** – A smaller slice (32 %) comes from *last month* (AR 0.32), smoothing sudden spikes or dips rather than copying them outright.
• **Holiday adjustment** – If the month has a public holiday, the model subtracts roughly **8.8 k passengers** because leisure demand never fully replaces missing business trips.
• **Error “shock absorbers”** – The two negative MA numbers (-0.23 short-term, -0.38 seasonal) nudge forecasts down after an over-shoot and up after an under-shoot, so mistakes don’t snowball.
• **How good is it?** – With **R² = 0.94** and a typical miss of **±12.8 k** passengers (≈ 3 %), the model tracks history closely.
• **What to expect** – Given those errors and the random-looking residual checks, the **412 k-passenger** forecast is likely within **±17 k** unless unusual events disrupt the usual pattern.
""")

# ───────────────────────────────────────────────────────────────────────
for llm_label in AVAILABLE_LLM_LABELS:
    if llm_label == "DEEPSEEK":
        print(f"⏭️  Skipping CoT for {llm_label}")
        continue

    llm = get_llm(llm_label)
    print(f"\n### COT FEW-SHOT | LLM = {llm_label}")

    for (row_pos, model_name, xai_kind), human_msg_txt in HUMAN_PROMPTS_ALL.items():

        msgs = [
            SYSTEM_MSG,
            fewshot_h1, fewshot_a1,
            fewshot_h2, fewshot_a2,
            fewshot_h3, fewshot_a3,
            fewshot_h4, fewshot_a4,
            HumanMessage(content=human_msg_txt),
        ]

        t0   = time.time()
        resp = call_with_retry(llm, msgs)
        dt_s = time.time() - t0
        raw  = resp.content.strip()

        # split out optional <think> … </think>
        m          = THINK_TAG_RE.search(raw)
        internal   = m.group(1).strip() if m else "<none>"
        answer     = raw[m.end():].strip() if m else raw

        # core numbers ---------------------------------------------------
        true_val = float(y_test.iloc[row_pos])
        pred_val = float(preds[model_name][row_pos])
        pct_err  = (pred_val - true_val) / true_val * 100
        feat_vals = test.iloc[row_pos][FEATURES].to_dict()

        # XAI payload ----------------------------------------------------
        xai_vals, base_val = {}, None
        if xai_kind == "shap":
            vec       = SHAP_INFO[model_name]["values"][row_pos]
            xai_vals  = dict(zip(FEATURES, map(float, vec)))
            base_val  = float(SHAP_INFO[model_name]["base_value"])
        elif xai_kind == "lime":
            row_lime  = lime_full_df.loc[(model_name, test.index[row_pos])]
            base_val  = float(row_lime["intercept"])
            xai_vals  = row_lime.drop("intercept").to_dict()

        # save -----------------------------------------------------------
        add_row(STRATEGY_NAME, dict(
            LLM            = llm_label,
            Temperature    = "<default>",
            SampleID       = 1,
            RowPos         = row_pos,
            Model          = model_name,
            XAI            = xai_kind,
            WeekEndDate    = test.iloc[row_pos]["DateTime"],
            Prediction     = pred_val,
            TrueValue      = true_val,
            PctError       = pct_err,
            Duration_s     = dt_s,
            TokensThought  = n_tokens(internal),
            TokensAnswer   = n_tokens(answer),
            TokensTotal    = n_tokens(raw),
            MAE            = metrics_df.at[model_name, "MAE"],
            RMSE           = metrics_df.at[model_name, "RMSE"],
            R2             = metrics_df.at[model_name, "R2"],
            SystemMsg      = SYSTEM_MSG.content,
            HumanMsg       = human_msg_txt,
            InternalThought= internal,
            Explanation    = answer,
            XAI_base_value = base_val,
            Feature_vals   = feat_vals,
            XAI_vals       = xai_vals,
        ))

        print(f"  ✔ Row {row_pos:3d} | {model_name:<11} | {xai_kind:<4} | "
              f"{dt_s:5.2f}s | tok≈{n_tokens(raw)}")

# flush to CSV
save_csv(STRATEGY_NAME)
print(f"✅  Saved → explanations/{STRATEGY_NAME}.csv")


In [ ]:
# %% ─────────────────────────────────────────────────────────────────────
# 🏷️  ROLE-BASED STRATEGY  → explanations/role_based.csv
#     – Prepends a teacher-style role prompt to the standard system msg
# -----------------------------------------------------------------------
STRATEGY_NAME = "role_based"
THINK_RE = re.compile(r"<think>(.*?)</think>", re.I | re.S)

ROLE_SYSTEM_PROMPT = """
You are a seasoned data-scientist and approachable teacher.
You love turning complex numbers into everyday language so
home-owners can make smart, confident decisions.
""".strip()

for llm_label in AVAILABLE_LLM_LABELS:
    llm = get_llm(llm_label)
    print(f"\n### ROLE-BASED | LLM = {llm_label}")

    # system message for this strategy
    sys_msg = SystemMessage(
        content=system_prompt_template.strip() + "\n\n" + ROLE_SYSTEM_PROMPT
    )

    for (row_pos, model_name, xai_kind), human_txt in HUMAN_PROMPTS_ALL.items():

        msgs = [sys_msg, HumanMessage(content=human_txt)]

        t0   = time.time()
        resp = call_with_retry(llm, msgs)
        dt_s = time.time() - t0
        raw  = resp.content.strip()

        # optional <think> block (DeepSeek often returns one)
        m          = THINK_RE.search(raw)
        internal   = m.group(1).strip() if m else "<none>"
        answer     = raw[m.end():].strip() if m else raw

        # ------- numeric / context fields --------------------------------
        true_val  = float(y_test.iloc[row_pos])
        pred_val  = float(preds[model_name][row_pos])
        pct_err   = (pred_val - true_val) / true_val * 100
        feat_vals = test.iloc[row_pos][FEATURES].to_dict()

        # ------- XAI payload ---------------------------------------------
        xai_vals, base_val = {}, None
        if xai_kind == "shap":
            vec       = SHAP_INFO[model_name]["values"][row_pos]
            xai_vals  = dict(zip(FEATURES, map(float, vec)))
            base_val  = float(SHAP_INFO[model_name]["base_value"])
        elif xai_kind == "lime":
            row_lime  = lime_full_df.loc[(model_name, test.index[row_pos])]
            base_val  = float(row_lime["intercept"])
            xai_vals  = row_lime.drop("intercept").to_dict()

        # ------- save row -------------------------------------------------
        add_row(STRATEGY_NAME, dict(
            LLM            = llm_label,
            Temperature    = "<default>",
            SampleID       = 1,
            RowPos         = row_pos,
            Model          = model_name,
            XAI            = xai_kind,
            WeekEndDate    = test.iloc[row_pos]["DateTime"],
            Prediction     = pred_val,
            TrueValue      = true_val,
            PctError       = pct_err,
            Duration_s     = dt_s,
            TokensThought  = n_tokens(internal),
            TokensAnswer   = n_tokens(answer),
            TokensTotal    = n_tokens(raw),
            MAE            = metrics_df.at[model_name, "MAE"],
            RMSE           = metrics_df.at[model_name, "RMSE"],
            R2             = metrics_df.at[model_name, "R2"],
            SystemMsg      = sys_msg.content,
            HumanMsg       = human_txt,
            InternalThought= internal,
            Explanation    = answer,
            XAI_base_value = base_val,
            Feature_vals   = feat_vals,
            XAI_vals       = xai_vals,
        ))

        print(f"  ✔ Row {row_pos:3d} | {model_name:<11} | {xai_kind:<4} | "
              f"{dt_s:5.2f}s | tok≈{n_tokens(raw)}")

# flush to CSV
save_csv(STRATEGY_NAME)
print(f"✅  Saved → explanations/{STRATEGY_NAME}.csv")


In [ ]:
# %% ─────────────────────────────────────────────────────────────────────
# 🧩  META-PROMPTING STRATEGY  → explanations/meta_prompting.csv
#     – Uses its own system prompt (does NOT overwrite system_prompt_template)
# -----------------------------------------------------------------------

STRATEGY_NAME = "meta_prompting"
THINK_RE = re.compile(r"<think>(.*?)</think>", re.I | re.S)

# ----- 1. dedicated system prompt --------------------------------------
META_SYSTEM_PROMPT = """
Carefully interpret the time-series forecasting context that follows.

Your goal is to help a non-technical lay user with no background in statistical
models or explainability methods such as SHAP or LIME.  They need to understand
• **why** the model produced this forecast, and
• **how much confidence** they can place in it.

OUTPUT RULES
• Write **exactly six bullet points** — no more, no less.
• Keep the whole response **≤ 200 words**.
• Use plain language, but key TS terms like *lag*, *trend*, *seasonality*,
  *baseline*, *error* are fine.
• Do **not** reveal code.

Deliver a clear, concise explanation that lets the reader judge the forecast’s
plausibility and reliability.
""".strip()

# ----- 2. meta-prompt prefix (goes before each row-specific prompt) -----
META_HUMAN_PREFIX = """\
Let's think step by step:

- **Context**: Summarise the problem domain and briefly name the XAI method.
- **Data Insights**: Note dataset size, key features, possible biases.
- **Model/Parameters**: Mention any parameter that affects performance or interpretability.
- **Performance Metrics**: Explain MAE, RMSE, R².
- **XAI / Feature Importance**:
    1) Top contributing factors.
    2) Why they matter.
    3) Surprising correlations (if any).
- **Missing Info**: Acknowledge gaps without speculation.

**Below is the information**
"""

# -----------------------------------------------------------------------
for llm_label in AVAILABLE_LLM_LABELS:
    llm = get_llm(llm_label)
    print(f"\n### META-PROMPTING | LLM = {llm_label}")

    sys_msg = SystemMessage(content=META_SYSTEM_PROMPT)

    for (row_pos, model_name, xai_kind), human_txt in HUMAN_PROMPTS_ALL.items():

        meta_human = HumanMessage(
            content=META_HUMAN_PREFIX + "\n\n" + human_txt + "\n"
        )

        msgs = [sys_msg, meta_human]

        t0   = time.time()
        resp = call_with_retry(llm, msgs)
        dt_s = time.time() - t0
        raw  = resp.content.strip()

        m          = THINK_RE.search(raw)
        internal   = m.group(1).strip() if m else "<none>"
        answer     = raw[m.end():].strip() if m else raw

        # ---------- numeric / context bits ------------------------------
        true_val  = float(y_test.iloc[row_pos])
        pred_val  = float(preds[model_name][row_pos])
        pct_err   = (pred_val - true_val) / true_val * 100
        feat_vals = test.iloc[row_pos][FEATURES].to_dict()

        # ---------- XAI payload -----------------------------------------
        xai_vals, base_val = {}, None
        if xai_kind == "shap":
            vec       = SHAP_INFO[model_name]["values"][row_pos]
            xai_vals  = dict(zip(FEATURES, map(float, vec)))
            base_val  = float(SHAP_INFO[model_name]["base_value"])
        elif xai_kind == "lime":
            row_lime  = lime_full_df.loc[(model_name, test.index[row_pos])]
            base_val  = float(row_lime["intercept"])
            xai_vals  = row_lime.drop("intercept").to_dict()

        # ---------- save row via helper ---------------------------------
        add_row(STRATEGY_NAME, dict(
            LLM            = llm_label,
            Temperature    = "<default>",
            SampleID       = 1,
            RowPos         = row_pos,
            Model          = model_name,
            XAI            = xai_kind,
            WeekEndDate    = test.iloc[row_pos]["DateTime"],
            Prediction     = pred_val,
            TrueValue      = true_val,
            PctError       = pct_err,
            Duration_s     = dt_s,
            TokensThought  = n_tokens(internal),
            TokensAnswer   = n_tokens(answer),
            TokensTotal    = n_tokens(raw),
            MAE            = metrics_df.at[model_name, "MAE"],
            RMSE           = metrics_df.at[model_name, "RMSE"],
            R2             = metrics_df.at[model_name, "R2"],
            SystemMsg      = META_SYSTEM_PROMPT,
            HumanMsg       = meta_human.content,
            InternalThought= internal,
            Explanation    = answer,
            XAI_base_value = base_val,
            Feature_vals   = feat_vals,
            XAI_vals       = xai_vals,
        ))

        print(f"  ✔ Row {row_pos:3d} | {model_name:<11} | {xai_kind:<4} | "
              f"{dt_s:5.2f}s | tok≈{n_tokens(raw)}")

# flush to CSV
save_csv(STRATEGY_NAME)
print(f"✅  Saved → explanations/{STRATEGY_NAME}.csv")


In [ ]:
# %% ─────────────────────────────────────────────────────────────────────
# 🤔  REFLEXION STRATEGY  → explanations/reflexion.csv
#      (works for SHAP • LIME • none • SARIMAX)
# -----------------------------------------------------------------------
import re, json, textwrap, pprint, time
from langchain.schema import SystemMessage, HumanMessage, AIMessage

STRATEGY_NAME   = "reflexion"
MAX_TRIALS      = 6           # drafts per prompt
MEMORY_SIZE     = 3           # last N self-reflections kept
THINK_RE        = re.compile(r"<think>(.*?)</think>", re.I | re.S)   # DeepSeek guard

# ───────────────────────────────────────────────────────────────────────
# Helper: parse prediction + either SHAP or LIME section (if present)
# ───────────────────────────────────────────────────────────────────────
_XAI_BLOCK_RE = re.compile(
    r"(?P<label>SHAP|LIME) values:\s*([\s\S]+?)\n(?=[A-Za-z_-]+:|The expected/base)",
    re.I,
)

def parse_pred_and_xai(human_prompt: str):
    """Return (prediction, {feat: value}, label_str or '<none>')."""
    pred_match = re.search(r"Prediction:\s*([\d.]+)", human_prompt)
    pred = float(pred_match.group(1)) if pred_match else None

    xai_vals = {}
    xai_label = "<none>"

    m = _XAI_BLOCK_RE.search(human_prompt)
    if m:
        xai_label = m.group("label").upper()
        for f, v in re.findall(r"([A-Za-z0-9_]+):\s*([+-]?[\d.]+)", m.group(0)):
            xai_vals[f] = float(v)

    return pred, xai_vals, xai_label


# ───────────────────────────────────────────────────────────────────────
# Helper: grader prompt (adapts if no XAI info)
# ───────────────────────────────────────────────────────────────────────
def llm_grade(expl_text: str, pred: float,
              xai_vals: dict[str, float], top_k: int = 5):
    """
    Ask GPT-4-o (hard-wired) to grade the explanation.
    Returns (passed_bool, raw_json_line, scores_dict)
    – Pass if accuracy≥4 & coverage≥4 & clarity≥4.
    """
    llm_grader = get_llm("GPT")   # we always grade with GPT-4o

    if xai_vals:
        top_feats  = sorted(xai_vals, key=lambda f: abs(xai_vals[f]),
                             reverse=True)[:top_k]
        top_lines  = "\n".join(f"{f}: {xai_vals[f]:.3f}" for f in top_feats)
        facts_block = f"Prediction: {pred}\nTop features ± values:\n{top_lines}"
    else:
        facts_block = f"Prediction: {pred}\n(no XAI information supplied)"

    rubric_prompt = textwrap.dedent(f"""
        You are evaluating a time-series forecast explanation for a lay reader.

        **Ground-truth facts**
        {facts_block}

        **Explanation to grade**
        <<<
        {expl_text}
        >>>

        Rate each criterion 1-5 (briefly justify ≤15 words):
        1. Accuracy / Faithfulness
        2. Coverage of key drivers
        3. Confidence communication
        4. Clarity for a lay reader

        Respond with *one* valid JSON line:
        {{"accuracy": x, "coverage": y, "confidence": z, "clarity": w,
          "justifications": {{"accuracy": "…" }} }}
    """)

    grade_raw = call_with_retry(llm_grader, [HumanMessage(content=rubric_prompt)])
    clean = re.sub(r"```(?:json)?|```", "", grade_raw.content).strip()
    try:
        scores = json.loads(clean)
    except json.JSONDecodeError:
        scores = {"accuracy": 0, "coverage": 0, "confidence": 0, "clarity": 0}

    passed = (
        scores.get("accuracy", 0)   >= 4 and
        scores.get("coverage", 0)   >= 4 and
        scores.get("clarity", 0)    >= 4
    )
    return passed, clean, scores


# ───────────────────────────────────────────────────────────────────────
# Main loop over every prepared prompt
# ───────────────────────────────────────────────────────────────────────
for llm_label in AVAILABLE_LLM_LABELS:

    llm_actor = get_llm(llm_label)
    print(f"\n### REFLEXION | LLM = {llm_label}")

    # Iterate over *all* prompts (row, model, xai_kind)
    for key, human_txt in HUMAN_PROMPTS_ALL.items():
        row_pos, model_name, xai_kind = key
        true_val  = float(y_test.iloc[row_pos])
        pred_val  = float(preds[model_name][row_pos])
        pct_err   = (pred_val - true_val) / true_val * 100
        feat_vals = test.iloc[row_pos][FEATURES].to_dict()

        # ground-truth XAI dict for grader (may be empty)
        if xai_kind == "shap":
            vec = SHAP_INFO[model_name]["values"][row_pos]
            gt_xai = dict(zip(FEATURES, map(float, vec)))
            base_val = float(SHAP_INFO[model_name]["base_value"])
        elif xai_kind == "lime":
            row_lime  = lime_full_df.loc[(model_name, test.index[row_pos])]
            gt_xai    = row_lime.drop("intercept").to_dict()
            base_val  = float(row_lime["intercept"])
        else:          # none
            gt_xai, base_val = {}, None

        # Parse again for safety (prediction & XAI from *prompt text*)
        pred_in_prompt, _, _ = parse_pred_and_xai(human_txt)

        # ---------- Reflexion loop per prompt ------------------------
        memory      : list[str] = []
        tokens_drafts = tokens_critics = tokens_reflect = 0
        start_t = time.time()

        for episode in range(1, MAX_TRIALS + 1):
            # 3.1  draft
            actor_msgs = (
                [SystemMessage(content=system_prompt_template)] +
                [AIMessage(content=m) for m in memory] +
                [HumanMessage(content=human_txt)]
            )
            draft_resp = call_with_retry(llm_actor, actor_msgs)
            draft_text = draft_resp.content.strip()
            tokens_drafts += n_tokens(draft_text)

            # 3.2 grade
            passed, grade_json, scores = llm_grade(
                draft_text,
                pred_in_prompt or pred_val,
                gt_xai
            )
            tokens_critics += n_tokens(grade_json)

            # 3.3 reflection
            reflect_prompt = (
                f"Outcome: {'SUCCESS' if passed else 'FAIL'}.\n"
                "Give ONE short sentence advising yourself how to improve "
                "accuracy, coverage, confidence, clarity next time."
            )
            reflect_resp = call_with_retry(llm_actor, [HumanMessage(content=reflect_prompt)])
            reflection = reflect_resp.content.strip()
            tokens_reflect += n_tokens(reflection)

            memory.append(reflection)
            memory = memory[-MEMORY_SIZE:]

            if passed:
                break  # success early

        total_time = time.time() - start_t
        final_expl = draft_text  # from last iteration (pass or max-trials)

        # internal thought split (if any)
        m = THINK_RE.match(final_expl)
        internal = m.group(1).strip() if m else "<none>"
        answer   = final_expl[m.end():].strip() if m else final_expl

        # ------------- save row --------------------------------------
        add_row(STRATEGY_NAME, dict(
            LLM            = llm_label,
            Temperature    = "<default>",
            SampleID       = 1,
            RowPos         = row_pos,
            Model          = model_name,
            XAI            = xai_kind,
            WeekEndDate    = test.iloc[row_pos]["DateTime"],
            Prediction     = pred_val,
            TrueValue      = true_val,
            PctError       = pct_err,
            Duration_s     = total_time,
            TokensThought  = tokens_reflect,        # self reflections
            TokensDrafts   = tokens_drafts,
            TokensCritic   = tokens_critics,
            TokensAnswer   = n_tokens(answer),
            TokensTotal    = (tokens_reflect + tokens_drafts +
                              tokens_critics + n_tokens(answer)),
            MAE            = metrics_df.at[model_name, "MAE"],
            RMSE           = metrics_df.at[model_name, "RMSE"],
            R2             = metrics_df.at[model_name, "R2"],
            SystemMsg      = system_prompt_template,
            HumanMsg       = human_txt,
            InternalThought= internal,
            Explanation    = answer,
            GraderJSON     = grade_json,
            XAI_base_value = base_val,
            Feature_vals   = feat_vals,
            XAI_vals       = gt_xai,
        ))

        print(f"  ✔ Row {row_pos:3d} | {model_name:<11} | {xai_kind:<4} | "
              f"{total_time:5.1f}s | pass={passed}")

# ── finally write the CSV ----------------------------------------------
save_csv(STRATEGY_NAME)


In [ ]:
# %% ─────────────────────────────────────────────────────────────────────
# 🔁  SELF-CONSISTENCY  (7 drafts → 1 synthesis)  → explanations/self_consistency.csv
# -----------------------------------------------------------------------
import time, textwrap, json, re
from langchain.schema import SystemMessage, HumanMessage

STRATEGY_NAME   = "self_consistency"
N_CANDIDATES    = 7
THINK_TAG_RE    = re.compile(r"<think>(.*?)</think>", re.I | re.S)

SYSTEM_MSG_TXT  = system_prompt_template.strip()
SYSTEM_MSG      = SystemMessage(content=SYSTEM_MSG_TXT)

for llm_label in AVAILABLE_LLM_LABELS:          # includes DeepSeek
    llm = get_llm(llm_label)
    llm_synth    = get_llm(llm_label, temperature=0.2)
    print(f"\n### SELF-CONSISTENCY | LLM = {llm_label}")

    for (row_pos, model_name, xai_kind), human_txt in HUMAN_PROMPTS_ALL.items():

        # ── collect 7 individual drafts ────────────────────────────────
        drafts     = []
        tok_drafts = 0
        t0_total   = time.time()

        base_msgs = [SYSTEM_MSG, HumanMessage(content=human_txt)]

        for _ in range(N_CANDIDATES):
            resp = call_with_retry(llm, base_msgs)
            draft = resp.content.strip()
            drafts.append(draft)
            tok_drafts += n_tokens(draft)

        # ── build the synthesis prompt & run once ──────────────────────
        synth_prompt = (
            "Below are multiple candidate explanations produced by the model for the same input. "
            "Please create a single, cohesive explanation that:\n"
            "  - Retains the most insightful sentences from each.\n"
            "  - Resolves contradictions.\n"
            "  - Still obeys the format instructions.\n\n"
            "=== CANDIDATES ===\n" +
            "\n".join(f"--- Candidate #{i+1} ---\n{c}" for i, c in enumerate(drafts))
        )
        synth_msgs   = [SYSTEM_MSG, HumanMessage(content=synth_prompt)]
        synth_resp   = call_with_retry(llm_synth, synth_msgs)
        synth_text   = synth_resp.content.strip()
        tok_synth    = n_tokens(synth_text)
        elapsed_s    = time.time() - t0_total

        # optional <think> removal (DeepSeek etc.)
        m = THINK_TAG_RE.match(synth_text)
        internal = m.group(1).strip() if m else "<none>"
        answer   = synth_text[m.end():].strip() if m else synth_text

        # ── numeric context -------------------------------------------------
        true_val  = float(y_test.iloc[row_pos])
        pred_val  = float(preds[model_name][row_pos])
        pct_err   = (pred_val - true_val) / true_val * 100
        feat_vals = test.iloc[row_pos][FEATURES].to_dict()

        # XAI specifics
        xai_vals, base_val = {}, None
        if xai_kind == "shap":
            vec      = SHAP_INFO[model_name]["values"][row_pos]
            xai_vals = dict(zip(FEATURES, map(float, vec)))
            base_val = float(SHAP_INFO[model_name]["base_value"])
        elif xai_kind == "lime":
            row_lime = lime_full_df.loc[(model_name, test.index[row_pos])]
            base_val = float(row_lime["intercept"])
            xai_vals = row_lime.drop("intercept").to_dict()

        # store
        add_row(STRATEGY_NAME, dict(
            LLM             = llm_label,
            Temperature     = "<default>",
            SampleID        = 1,
            RowPos          = row_pos,
            Model           = model_name,
            XAI             = xai_kind,
            WeekEndDate     = test.iloc[row_pos]["DateTime"],
            Prediction      = pred_val,
            TrueValue       = true_val,
            PctError        = pct_err,
            Duration_s      = elapsed_s,
            TokensDrafts    = tok_drafts,
            TokensSynth     = tok_synth,
            TokensAnswer    = n_tokens(answer),
            TokensTotal     = tok_drafts + tok_synth,
            SystemMsg       = SYSTEM_MSG_TXT,
            HumanMsg        = human_txt,
            CandidatesJSON  = json.dumps(drafts, ensure_ascii=False),
            InternalThought = internal,
            Explanation     = answer,
            XAI_base_value  = base_val,
            Feature_vals    = feat_vals,
            XAI_vals        = xai_vals,
            MAE             = metrics_df.at[model_name, "MAE"],
            RMSE            = metrics_df.at[model_name, "RMSE"],
            R2              = metrics_df.at[model_name, "R2"],
        ))

        print(f"  ✔ Row {row_pos:3d} | {model_name:<11} | {xai_kind:<4} | "
              f"{elapsed_s:5.1f}s | tokDrafts={tok_drafts}")

# final flush
save_csv(STRATEGY_NAME)
